# Machine Learning per la Valutazione e lo Stock Screening delle Banche Italiane Quotate

Notebook Colab riutilizzabile per testare fair value peer-implied, mispricing, screening bancario e expected returns su un panel `date`/`ticker` di banche italiane usando `ml_stock_lab`.

## 1. Setup & imports

Se esegui in Colab e `ml_stock_lab` non e' gia' disponibile, installa o carica il package prima di eseguire gli import. Nel caso di repository clonato, puoi aggiungere la root del repo al `PYTHONPATH`.

In [1]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    candidates = []
    env_root = os.environ.get('RESEARCH_PLATFORM_ROOT') or os.environ.get('PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        Path('/content/drive/MyDrive/GitHub/machine-learning-for-trading'),
        Path('/content/drive/MyDrive/machine-learning-for-trading'),
        Path('/content/machine-learning-for-trading'),
        Path.home() / 'Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/GitHub/machine-learning-for-trading',
        Path.home() / 'GitHub/machine-learning-for-trading',
        cwd,
        *cwd.parents,
    ])
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        if (candidate / 'ml_stock_lab').exists() or (candidate / 'research_platform_definitive' / 'src').exists():
            return candidate.resolve()
    return cwd


PROJECT_ROOT = _find_repo_root()
# Root package `ml_stock_lab/` is the paper API; keep it before the definitive src package.
for repo_path in [PROJECT_ROOT / 'research_platform_definitive' / 'src', PROJECT_ROOT / 'research_platform_definitive', PROJECT_ROOT]:
    if repo_path.exists():
        path_str = str(repo_path)
        if path_str in sys.path:
            sys.path.remove(path_str)
        sys.path.insert(0, path_str)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display, HTML, clear_output
except Exception:
    def display(x=None, *args, **kwargs):
        if x is not None:
            print(x)
    HTML = lambda x: x
    def clear_output(*args, **kwargs):
        return None

from ml_stock_lab.datasets import load_panel_csv, build_company_selection_panel, build_company_selection_widget
from ml_stock_lab.features import fundamentals, technical
from ml_stock_lab.valuation import (
    PeerImpliedOLS,
    PeerImpliedLasso,
    PeerImpliedRF,
    PeerImpliedGBRT,
)
from ml_stock_lab.signals import MispricingSignal, EnsembleMispricingSignal
from ml_stock_lab.screening import ScreeningFunction, TopNSelector, QuantileSorter
from ml_stock_lab.prediction import FundamentalPredictorRF, ExpectedReturnModel, oos_r2
from ml_stock_lab.portfolio import QuantilePortfolioBuilder, LongShortPortfolioBuilder
from ml_stock_lab.evaluation import (
    PerformanceMetrics,
    FactorAlphaEvaluator,
    plot_cumulative_returns,
)

try:
    from research_platform_core import (
        ITALIAN_LISTED_BANK_TICKERS as CORE_ITALIAN_BANK_TICKERS,
        build_banks_universe,
        build_yfinance_bank_panel as core_build_yfinance_bank_panel,
        run_banks_data_pipeline,
    )
    BANKING_DATA_ENGINE_AVAILABLE = True
except Exception as exc:
    BANKING_DATA_ENGINE_AVAILABLE = False
    CORE_ITALIAN_BANK_TICKERS = {}
    build_banks_universe = None
    core_build_yfinance_bank_panel = None
    run_banks_data_pipeline = None
    print(f'Banking data engine non disponibile: {exc}')

try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    px = None
    PLOTLY_AVAILABLE = False

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)
pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 180)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


PROJECT_ROOT: /Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/GitHub/machine-learning-for-trading


## 2. Configurazione esperimento

### Universo, target e peer group

Il **target** è la banca che vuoi osservare con più attenzione nel lab (per esempio `ISP.MI`). Il **peer/universe** è invece il gruppo di banche usato per stimare il fair value peer-implied, ordinare i segnali e costruire i portafogli per quantile. In pratica: il target è il nome da studiare, mentre l'universo è il contesto competitivo e statistico entro cui il modello valuta quel nome.

L'universo base viene costruito da `company_panel` selezionando società con `sector == "Banks"` e `country` italiano (`IT` o `Italy`), con override manuali per wealth/private banking italiani come Fineco, Mediolanum, Banca Generali e CREDEM quando disponibili nei dati. Questo evita di mischiare banche italiane con assicurazioni, industriali o altri financial non comparabili, ma lascia comunque la possibilità di aggiungere ticker manuali.

La motivazione economica dell'universo allargato è semplice: le grandi banche FTSE MIB (Intesa, UniCredit, Banco BPM, BPER, MPS) danno profondità e liquidità; mid-cap e wealth managers (Fineco, Mediolanum, Banca Generali, CREDEM) aggiungono modelli di business più fee-based e meno puramente creditizi; peer europei opzionali possono servire come benchmark, ma vanno usati con cautela perché regolazione, mix ricavi e ciclo macro possono differire.


In [2]:
# ============================================================
# User-friendly experiment configuration
# ============================================================

WEALTH_BANK_TICKERS = ['BMED.MI', 'BGN.MI', 'FBK.MI', 'CE.MI']
ITALIAN_COUNTRY_KEYS = {'IT', 'ITALY', 'ITALIA'}

DEFAULT_SELECTED_TICKERS = [
    'ISP.MI', 'UCG.MI', 'BAMI.MI', 'BPE.MI', 'BMPS.MI',
    'CE.MI', 'BGN.MI', 'FBK.MI', 'BMED.MI', 'MB.MI',
]

ITALIAN_BANK_UNIVERSES = {
    'Major Italian Banks': ['ISP.MI', 'UCG.MI', 'BAMI.MI', 'BPE.MI', 'BMPS.MI', 'MB.MI'],
    'Italian Banks + Wealth': ['ISP.MI', 'UCG.MI', 'BAMI.MI', 'BPE.MI', 'BMPS.MI', 'CE.MI', 'BGN.MI', 'FBK.MI', 'BMED.MI', 'MB.MI'],
    'Large Liquid Italy': ['ISP.MI', 'UCG.MI', 'BAMI.MI', 'BPE.MI', 'FBK.MI', 'MB.MI'],
    'Manual': DEFAULT_SELECTED_TICKERS,
    'Euro Area ECB peers': DEFAULT_SELECTED_TICKERS,  # expanded dynamically after ECB universe refresh
}

FAIR_VALUE_MODEL_MAP = {
    'OLS': PeerImpliedOLS,
    'LASSO': PeerImpliedLasso,
    'Random Forest': PeerImpliedRF,
    'Gradient Boosting': PeerImpliedGBRT,
}

PROJECT_DATA_DIR = PROJECT_ROOT / 'papers' / 'italian_banks_ml_stock_screening' / 'data'
BANKS_PIPELINE_OUTPUT_DIR = PROJECT_ROOT / 'papers' / 'italian_banks_ml_stock_screening' / 'output' / 'banks_pipeline'
BANKS_PIPELINE_CACHE_DIR = PROJECT_DATA_DIR / '_banking_cache'
DRIVE_DB_ROOT = Path(os.environ.get('FINANCIAL_DB_ROOT', '/content/drive/MyDrive/Database Finanziario')).expanduser()

DATA_PATH = None
CANDIDATE_DATA_PATHS = [
    PROJECT_DATA_DIR / 'italian_banks_panel.csv',
    PROJECT_ROOT / 'italian_banks_panel.csv',
    PROJECT_ROOT / 'research_platform_definitive' / 'output' / 'ml_stock_lab' / 'tables' / 'MLStockLab_panel.csv',
    DRIVE_DB_ROOT / 'italian_banks_panel.csv',
    DRIVE_DB_ROOT / 'papers' / 'italian_banks_ml_stock_screening' / 'italian_banks_panel.csv',
    Path('/content/italian_banks_panel.csv'),
]

USER_CONFIG = {
    'data_mode': 'auto',               # auto, csv_only, yfinance_refresh
    'force_refresh': False,
    'use_api_price_fallbacks': True,
    'api_provider_order': ['eodhd', 'finnhub', 'alpha_vantage', 'tiingo', 'polygon', 'yfinance'],
    'save_generated_panel': True,
    'universe_name': 'Italian Banks + Wealth',
    'universe_scope': 'Italy only',
    'target_ticker': 'ISP.MI',
    'selected_tickers': ITALIAN_BANK_UNIVERSES['Italian Banks + Wealth'],
    'manual_peers': ['UCG.MI', 'BAMI.MI', 'BPE.MI', 'MB.MI', 'FBK.MI'],
    'peer_method': 'manual_or_selected',
    'train_end': '2021-12-31',
    'test_start': '2022-01-31',
    'start_date': '2015-01-01',
    'end_date': None,
    'price_frequency': 'monthly',
    'fair_value_models': ['OLS', 'LASSO', 'Random Forest', 'Gradient Boosting'],
    'primary_model': 'Random Forest',
    'use_ensemble_mispricing': True,
    'rf_params': {'n_estimators': 160, 'max_depth': None, 'min_samples_leaf': 2},
    'gbrt_params': {'n_estimators': 160, 'max_depth': 3, 'min_samples_leaf': 2, 'learning_rate': 0.05},
    'expected_return_rf_params': {'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 2},
    'expected_return_model_specs': {
        'rf': {'enabled': True, 'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 2},
        'gbrt': {'enabled': True, 'n_estimators': 160, 'max_depth': 3, 'min_samples_leaf': 2, 'learning_rate': 0.05},
        'xgb': {'enabled': True, 'n_estimators': 160, 'max_depth': 3, 'learning_rate': 0.05},
        'mlp': {'enabled': True, 'hidden_layer_sizes': (32, 16), 'alpha': 0.001, 'max_iter': 800},
        'lasso': {'enabled': True, 'alpha': 0.001},
    },
    'n_quantiles': 5,
    'long_quantile': 5,
    'short_quantile': 1,
    'portfolio_weighting': 'equal',
    'hybrid_lambda': 0.55,
    'cet1_min': 11.0,
    'npl_max': 0.08,
    'liq_min': 0.0,
    'min_feature_non_null_ratio': 0.25,
    'fallback_to_market_features': True,
    'use_yfinance_info': True,
    'refresh_banks_universe': True,
    'include_ecb_macro': False,  # set True only when you want ECB SDMX calls during the run
    'bds_exports': {},           # add Banca d'Italia BDS CSV/XLSX export links here
    'external_fundamentals_path': None,
}

TRAIN_END = USER_CONFIG['train_end']
TEST_START = USER_CONFIG['test_start']
UNIVERSE_FILTER = {'country': 'IT', 'sector': 'Banks'}

BANK_FUNDAMENTAL_FEATURES = [
    'fund_roa_lag', 'fund_roe_lag', 'fund_nim_lag', 'fund_cost_income_lag',
    'fund_ldr_lag', 'fund_npl_ratio_lag', 'fund_cet1_lag', 'fund_assets_lag',
]
MARKET_FEATURES = [
    'mkt_pb', 'mkt_pe', 'mkt_beta_bank', 'mkt_vol_1y', 'mkt_mom_12m', 'mkt_mom_6m', 'mkt_drawdown_1y',
]
FAIR_VALUE_FEATURES = BANK_FUNDAMENTAL_FEATURES + MARKET_FEATURES
EXPECTED_RETURN_FEATURES = FAIR_VALUE_FEATURES + ['mispricing_z', 'mispricing_ens']

FORWARD_RETURN_COL = 'target_ret_1m_fwd'
LOG_MCAP_COL = 'target_log_mcap'
MARKET_CAP_COL = 'mkt_market_cap'
PRICE_COL = 'mkt_price'

SCREENING_PARAMS = {
    'cet1_min': USER_CONFIG['cet1_min'],
    'npl_max': USER_CONFIG['npl_max'],
    'liq_min': USER_CONFIG['liq_min'],
}

if BANKING_DATA_ENGINE_AVAILABLE and CORE_ITALIAN_BANK_TICKERS:
    for name, ticker in CORE_ITALIAN_BANK_TICKERS.items():
        if ticker not in ITALIAN_BANK_UNIVERSES['Italian Banks + Wealth']:
            ITALIAN_BANK_UNIVERSES['Italian Banks + Wealth'].append(ticker)


def _parse_tickers(text_or_list):
    if isinstance(text_or_list, (list, tuple, set)):
        raw = list(text_or_list)
    else:
        raw = str(text_or_list or '').replace(';', ',').replace('\n', ',').split(',')
    out = []
    for item in raw:
        ticker = str(item).strip().upper()
        if ticker and ticker not in out:
            out.append(ticker)
    return out




def build_italian_banks_universe(company_panel: pd.DataFrame) -> pd.DataFrame:
    """
    Restituisce un sottoinsieme di company_panel contenente l'universo 'banche italiane allargato'.
    Regole: sector == 'Banks', country in {'IT', 'Italy'}, più override wealth/private banking.
    Aggiunge una colonna universe_flag al panel di ritorno.
    """
    if company_panel is None or company_panel.empty:
        return pd.DataFrame(columns=list(getattr(company_panel, 'columns', [])) + ['universe_flag'])
    out = company_panel.copy()
    ticker = out.get('ticker', pd.Series('', index=out.index)).astype(str).str.upper().str.strip()
    sector = out.get('sector', pd.Series('', index=out.index)).astype(str).str.upper().str.strip()
    country = out.get('country', pd.Series('', index=out.index)).astype(str).str.upper().str.strip()
    is_bank = sector.eq('BANKS') | sector.str.contains('BANK', na=False)
    is_italy = country.isin(ITALIAN_COUNTRY_KEYS)
    is_override = ticker.isin(WEALTH_BANK_TICKERS + DEFAULT_SELECTED_TICKERS)
    out['universe_flag'] = np.where((is_bank & is_italy) | is_override, 'italian_banks_extended', '')
    return out[out['universe_flag'].ne('')].sort_values(['selected', 'ticker'], ascending=[False, True]).reset_index(drop=True)


def summarize_experiment_config(config: dict) -> pd.DataFrame:
    """Restituisce una tabella 1xN con i parametri chiave dell'esperimento."""
    row = {
        'universe': config.get('universe_name'),
        'scope': config.get('universe_scope'),
        'force_refresh': config.get('force_refresh'),
        'api_fallbacks': config.get('use_api_price_fallbacks'),
        'n_tickers': len(config.get('selected_tickers', [])),
        'target': config.get('target_ticker'),
        'primary_model': config.get('primary_model'),
        'ensemble': config.get('use_ensemble_mispricing'),
        'quantiles': config.get('n_quantiles'),
        'train_end': config.get('train_end'),
        'test_start': config.get('test_start'),
        'cet1_min': config.get('cet1_min'),
        'npl_max': config.get('npl_max'),
        'ldr_min': config.get('liq_min'),
        'hybrid_lambda': config.get('hybrid_lambda'),
        'rf_params': config.get('rf_params'),
        'gbrt_params': config.get('gbrt_params'),
        'expected_return_rf_params': config.get('expected_return_rf_params'),
    }
    return pd.DataFrame([row])


def _sync_experiment_globals():
    global TRAIN_END, TEST_START, QUANTILES, PRIMARY_MODEL_NAME, ENSEMBLE_ENABLED, RF_PARAMS, GBRT_PARAMS, EXPECTED_RETURN_RF_PARAMS, SCREENING_PARAMS
    TRAIN_END = USER_CONFIG['train_end']
    TEST_START = USER_CONFIG['test_start']
    QUANTILES = int(USER_CONFIG['n_quantiles'])
    PRIMARY_MODEL_NAME = USER_CONFIG.get('primary_model', 'Random Forest')
    ENSEMBLE_ENABLED = bool(USER_CONFIG.get('use_ensemble_mispricing', True))
    RF_PARAMS = dict(USER_CONFIG.get('rf_params', {}))
    GBRT_PARAMS = dict(USER_CONFIG.get('gbrt_params', {}))
    EXPECTED_RETURN_RF_PARAMS = dict(USER_CONFIG.get('expected_return_rf_params', {}))
    SCREENING_PARAMS = {'cet1_min': USER_CONFIG['cet1_min'], 'npl_max': USER_CONFIG['npl_max'], 'liq_min': USER_CONFIG['liq_min']}


def model_kwargs_for(model_name: str) -> dict:
    if model_name == 'Random Forest':
        return RF_PARAMS.copy()
    if model_name == 'Gradient Boosting':
        return GBRT_PARAMS.copy()
    return {}

def _render_config_cards(config):
    cards = [
        ('Data mode', config['data_mode']),
        ('Universe', f"{config['universe_name']} · {len(config['selected_tickers'])} tickers"),
        ('Target / peers', f"{config['target_ticker']} · {len(config['manual_peers'])} peers"),
        ('Models', ', '.join(config['fair_value_models'][:3]) + ('...' if len(config['fair_value_models']) > 3 else '')),
        ('Train/Test', f"<= {config['train_end']} / >= {config['test_start']}"),
        ('Screening', f"CET1>{config['cet1_min']} NPL<{config['npl_max']} LDR>{config['liq_min']}"),
    ]
    html = """
    <style>
    .ib-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:10px;margin:12px 0}
    .ib-card{background:#fff;border:1px solid #d9e2ec;border-radius:12px;padding:12px;box-shadow:0 1px 2px rgba(16,24,40,.04)}
    .ib-k{font-size:11px;text-transform:uppercase;color:#667085;font-weight:700}.ib-v{font-size:16px;color:#01696f;font-weight:760;margin-top:4px}
    .ib-note{background:#f6f8fb;border-left:5px solid #01696f;border-radius:8px;padding:12px;margin:10px 0;color:#344054}
    </style><div class='ib-grid'>
    """
    for k, v in cards:
        html += f"<div class='ib-card'><div class='ib-k'>{k}</div><div class='ib-v'>{v}</div></div>"
    html += '</div>'
    return html

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

if WIDGETS_AVAILABLE:
    style = {'description_width': '150px'}
    universe_w = widgets.Dropdown(options=list(ITALIAN_BANK_UNIVERSES), value=USER_CONFIG['universe_name'], description='Preset universo', style=style, layout=widgets.Layout(width='360px'))
    scope_w = widgets.ToggleButtons(options=['Italy only', 'Italy + EU banks'], value=USER_CONFIG['universe_scope'], description='Scope', style=style, layout=widgets.Layout(width='430px'))
    target_w = widgets.Combobox(options=sorted(set(sum(ITALIAN_BANK_UNIVERSES.values(), []))), value=USER_CONFIG['target_ticker'], description='Target ticker', ensure_option=False, style=style, layout=widgets.Layout(width='360px'))
    tickers_w = widgets.Textarea(value=', '.join(USER_CONFIG['selected_tickers']), description='Universo tickers', style=style, layout=widgets.Layout(width='760px', height='82px'))
    peers_w = widgets.Textarea(value=', '.join(USER_CONFIG['manual_peers']), description='Peer manuali', style=style, layout=widgets.Layout(width='760px', height='70px'))
    data_mode_w = widgets.Dropdown(options=['auto', 'csv_only', 'yfinance_refresh'], value=USER_CONFIG['data_mode'], description='Dati', style=style, layout=widgets.Layout(width='360px'))
    force_refresh_w = widgets.Checkbox(value=USER_CONFIG['force_refresh'], description='Force refresh datacenter/API', indent=False)
    api_fallback_w = widgets.Checkbox(value=USER_CONFIG['use_api_price_fallbacks'], description='Usa API price fallback da Colab secrets', indent=False)
    data_path_w = widgets.Text(value='' if DATA_PATH is None else str(DATA_PATH), description='CSV path', placeholder='Opzionale: percorso CSV custom', style=style, layout=widgets.Layout(width='760px'))
    save_panel_w = widgets.Checkbox(value=USER_CONFIG['save_generated_panel'], description='Salva panel generato in data/italian_banks_panel.csv', indent=False)
    start_w = widgets.Text(value=USER_CONFIG['start_date'], description='Start date', style=style, layout=widgets.Layout(width='260px'))
    train_end_w = widgets.Text(value=USER_CONFIG['train_end'], description='Train end', style=style, layout=widgets.Layout(width='260px'))
    test_start_w = widgets.Text(value=USER_CONFIG['test_start'], description='Test start', style=style, layout=widgets.Layout(width='260px'))
    models_w = widgets.SelectMultiple(options=list(FAIR_VALUE_MODEL_MAP), value=tuple(USER_CONFIG['fair_value_models']), description='Fair value models', style=style, layout=widgets.Layout(width='420px', height='120px'))
    primary_model_w = widgets.Dropdown(options=list(FAIR_VALUE_MODEL_MAP), value=USER_CONFIG['primary_model'], description='Primary model', style=style, layout=widgets.Layout(width='360px'))
    ensemble_w = widgets.Checkbox(value=USER_CONFIG['use_ensemble_mispricing'], description='Usa ensemble mispricing', indent=False)
    rf_trees_w = widgets.IntSlider(value=USER_CONFIG['rf_params']['n_estimators'], min=50, max=500, step=25, description='RF trees', style=style, layout=widgets.Layout(width='360px'))
    rf_depth_w = widgets.IntSlider(value=0 if USER_CONFIG['rf_params']['max_depth'] is None else USER_CONFIG['rf_params']['max_depth'], min=0, max=20, step=1, description='RF max depth', style=style, layout=widgets.Layout(width='360px'))
    rf_leaf_w = widgets.IntSlider(value=USER_CONFIG['rf_params']['min_samples_leaf'], min=1, max=20, step=1, description='RF min leaf', style=style, layout=widgets.Layout(width='360px'))
    gbrt_trees_w = widgets.IntSlider(value=USER_CONFIG['gbrt_params']['n_estimators'], min=50, max=500, step=25, description='GBRT trees', style=style, layout=widgets.Layout(width='360px'))
    gbrt_depth_w = widgets.IntSlider(value=USER_CONFIG['gbrt_params']['max_depth'], min=1, max=8, step=1, description='GBRT depth', style=style, layout=widgets.Layout(width='360px'))
    gbrt_leaf_w = widgets.IntSlider(value=USER_CONFIG['gbrt_params']['min_samples_leaf'], min=1, max=20, step=1, description='GBRT min leaf', style=style, layout=widgets.Layout(width='360px'))
    quantiles_w = widgets.IntSlider(value=USER_CONFIG['n_quantiles'], min=3, max=10, step=1, description='Quantili', style=style, layout=widgets.Layout(width='360px'))
    cet1_w = widgets.FloatSlider(value=USER_CONFIG['cet1_min'], min=7.0, max=18.0, step=0.25, description='CET1 min', style=style, layout=widgets.Layout(width='420px'))
    npl_w = widgets.FloatSlider(value=USER_CONFIG['npl_max'], min=0.01, max=0.25, step=0.005, description='NPL max', readout_format='.3f', style=style, layout=widgets.Layout(width='420px'))
    liq_w = widgets.FloatSlider(value=USER_CONFIG['liq_min'], min=0.0, max=2.0, step=0.05, description='LDR/liquidity min', style=style, layout=widgets.Layout(width='420px'))
    fallback_w = widgets.Checkbox(value=USER_CONFIG['fallback_to_market_features'], description='Se mancano fondamentali, usa feature tecniche/mercato', indent=False)
    hybrid_lambda_w = widgets.FloatSlider(value=USER_CONFIG['hybrid_lambda'], min=0.0, max=1.0, step=0.05, description='Hybrid lambda', style=style, layout=widgets.Layout(width='420px'))
    apply_w = widgets.Button(description='Applica configurazione esperimento', icon='check', button_style='success', layout=widgets.Layout(width='260px', height='42px'))
    out = widgets.Output()

    def _on_universe_change(change=None):
        tickers = ITALIAN_BANK_UNIVERSES.get(universe_w.value, DEFAULT_SELECTED_TICKERS)
        tickers_w.value = ', '.join(tickers)
        if target_w.value not in tickers:
            target_w.value = tickers[0]
        peers_w.value = ', '.join([t for t in tickers if t != target_w.value][:8])

    def _apply_config(_=None):
        global DATA_PATH, TRAIN_END, TEST_START, FAIR_VALUE_FEATURES, EXPECTED_RETURN_FEATURES, SCREENING_PARAMS
        with out:
            clear_output(wait=True)
            tickers = _parse_tickers(tickers_w.value)
            target = str(target_w.value or '').strip().upper() or (tickers[0] if tickers else 'ISP.MI')
            if target not in tickers:
                tickers = [target] + tickers
            peers = [p for p in _parse_tickers(peers_w.value) if p != target]
            USER_CONFIG.update({
                'data_mode': data_mode_w.value,
                'force_refresh': bool(force_refresh_w.value),
                'use_api_price_fallbacks': bool(api_fallback_w.value),
                'save_generated_panel': bool(save_panel_w.value),
                'universe_name': universe_w.value,
                'universe_scope': scope_w.value,
                'target_ticker': target,
                'selected_tickers': tickers,
                'manual_peers': peers or [t for t in tickers if t != target],
                'train_end': train_end_w.value,
                'test_start': test_start_w.value,
                'start_date': start_w.value,
                'fair_value_models': list(models_w.value) or ['Random Forest'],
                'primary_model': primary_model_w.value,
                'use_ensemble_mispricing': bool(ensemble_w.value),
                'rf_params': {'n_estimators': int(rf_trees_w.value), 'max_depth': None if int(rf_depth_w.value) == 0 else int(rf_depth_w.value), 'min_samples_leaf': int(rf_leaf_w.value)},
                'gbrt_params': {'n_estimators': int(gbrt_trees_w.value), 'max_depth': int(gbrt_depth_w.value), 'min_samples_leaf': int(gbrt_leaf_w.value), 'learning_rate': USER_CONFIG.get('gbrt_params', {}).get('learning_rate', 0.05)},
                'expected_return_rf_params': {'n_estimators': int(rf_trees_w.value), 'max_depth': None if int(rf_depth_w.value) == 0 else int(rf_depth_w.value), 'min_samples_leaf': int(rf_leaf_w.value)},
                'n_quantiles': int(quantiles_w.value),
                'long_quantile': int(quantiles_w.value),
                'short_quantile': 1,
                'cet1_min': float(cet1_w.value),
                'npl_max': float(npl_w.value),
                'liq_min': float(liq_w.value),
                'fallback_to_market_features': bool(fallback_w.value),
                'hybrid_lambda': float(hybrid_lambda_w.value),
            })
            DATA_PATH = Path(data_path_w.value).expanduser() if data_path_w.value.strip() else None
            _sync_experiment_globals()
            display(HTML(_render_config_cards(USER_CONFIG)))
            display(pd.DataFrame({'ticker': tickers, 'role': ['target' if t == target else 'peer/universe' for t in tickers]}))
            display(summarize_experiment_config(USER_CONFIG))

    universe_w.observe(_on_universe_change, names='value')
    apply_w.on_click(_apply_config)
    tabs = widgets.Tab(children=[
        widgets.VBox([widgets.HTML('<b>1. Universo e peers</b>'), widgets.HBox([universe_w, scope_w, target_w]), tickers_w, peers_w]),
        widgets.VBox([widgets.HTML('<b>2. Dati e fallback</b>'), widgets.HBox([data_mode_w, save_panel_w]), widgets.HBox([force_refresh_w, api_fallback_w]), data_path_w, widgets.HTML('<span style="color:#667085">Auto: CSV se esiste, altrimenti yfinance. csv_only: non usa API. yfinance_refresh: ricostruisce il panel.</span>')]),
        widgets.VBox([widgets.HTML('<b>3. Modelli e backtest</b>'), widgets.HBox([models_w, widgets.VBox([primary_model_w, quantiles_w, ensemble_w])]), widgets.HTML('<b>Hyperparams RF / GBRT</b>'), widgets.HBox([rf_trees_w, rf_depth_w, rf_leaf_w]), widgets.HBox([gbrt_trees_w, gbrt_depth_w, gbrt_leaf_w]), widgets.HBox([start_w, train_end_w, test_start_w])]),
        widgets.VBox([widgets.HTML('<b>4. Screening bancario</b>'), cet1_w, npl_w, liq_w, fallback_w, widgets.HTML('<b>Selection + timing</b>'), hybrid_lambda_w]),
    ])
    for idx, title in enumerate(['Universe & Peers', 'Data', 'Models', 'Screening']):
        tabs.set_title(idx, title)
    display(HTML("""
    <div style='background:#f6f8fb;border:1px solid #d9e2ec;border-left:5px solid #01696f;border-radius:10px;padding:14px;margin:10px 0'>
      <h3 style='margin:0;color:#01696f'>Italian Banks ML Lab · Control Center</h3>
      <p style='margin:6px 0 0;color:#344054'>Configura universo, peers, dati, modelli, split temporale, soglie regolamentari e fallback senza modificare codice.</p>
    </div>
    """))
    display(widgets.VBox([tabs, apply_w, out]))
    _on_universe_change(); _apply_config()
else:
    print('ipywidgets non disponibile: USER_CONFIG rimane modificabile via dizionario Python.')
    print(USER_CONFIG)


In [ ]:
# ============================================================
# Control Center watchdog / recovery cell
# ============================================================
from IPython.display import display, HTML

try:
    import ipywidgets as widgets
    IPYWIDGETS_AVAILABLE = True
except Exception as exc:
    IPYWIDGETS_AVAILABLE = False
    WIDGETS_IMPORT_ERROR = exc

if 'USER_CONFIG' not in globals():
    USER_CONFIG = {
        'universe_name': 'Italian Banks + Wealth',
        'selected_tickers': ['ISP.MI', 'UCG.MI', 'BAMI.MI', 'BPE.MI', 'BMPS.MI', 'MB.MI'],
        'target_ticker': 'ISP.MI',
        'primary_model': 'Random Forest',
        'n_quantiles': 5,
        'train_end': '2021-12-31',
        'test_start': '2022-01-31',
        'start_date': '2015-01-01',
        'cet1_min': 11.0,
        'npl_max': 0.08,
        'liq_min': 0.0,
        'use_ensemble_mispricing': True,
        'force_refresh': False,
        'universe_scope': 'Italy only',
    }


def debug_experiment_config(USER_CONFIG: dict) -> None:
    """
    Stampa un riepilogo chiaro della configurazione corrente dell'esperimento:
      - universo (tickers totali, tickers selezionati),
      - modello primario,
      - QUANTILES,
      - date start/train_end/test_start,
      - soglie CET1/NPL/LDR, eventuali flag ensemble/ML.
    """
    selected = USER_CONFIG.get('selected_tickers', []) or []
    if isinstance(selected, str):
        selected = _parse_tickers(selected) if '_parse_tickers' in globals() else [x.strip().upper() for x in selected.split(',') if x.strip()]
    rows = [
        {'field': 'universe_name', 'value': USER_CONFIG.get('universe_name')},
        {'field': 'universe_scope', 'value': USER_CONFIG.get('universe_scope')},
        {'field': 'target_ticker', 'value': USER_CONFIG.get('target_ticker')},
        {'field': 'selected_tickers_count', 'value': len(selected)},
        {'field': 'selected_tickers', 'value': ', '.join(selected[:16]) + (' ...' if len(selected) > 16 else '')},
        {'field': 'primary_model', 'value': USER_CONFIG.get('primary_model')},
        {'field': 'QUANTILES', 'value': USER_CONFIG.get('n_quantiles', globals().get('QUANTILES'))},
        {'field': 'start_date', 'value': USER_CONFIG.get('start_date')},
        {'field': 'train_end', 'value': USER_CONFIG.get('train_end')},
        {'field': 'test_start', 'value': USER_CONFIG.get('test_start')},
        {'field': 'cet1_min', 'value': USER_CONFIG.get('cet1_min')},
        {'field': 'npl_max', 'value': USER_CONFIG.get('npl_max')},
        {'field': 'liq_min', 'value': USER_CONFIG.get('liq_min')},
        {'field': 'use_ensemble_mispricing', 'value': USER_CONFIG.get('use_ensemble_mispricing')},
        {'field': 'force_refresh', 'value': USER_CONFIG.get('force_refresh')},
        {'field': 'use_api_price_fallbacks', 'value': USER_CONFIG.get('use_api_price_fallbacks')},
    ]
    display(HTML('<h4>Current experiment configuration</h4>'))
    display(pd.DataFrame(rows))


if IPYWIDGETS_AVAILABLE:
    universe_options = list(globals().get('ITALIAN_BANK_UNIVERSES', {'Manual': USER_CONFIG.get('selected_tickers', [])}).keys())
    universe_name = widgets.Dropdown(
        options=universe_options,
        value=USER_CONFIG.get('universe_name') if USER_CONFIG.get('universe_name') in universe_options else universe_options[0],
        description='Universe',
        layout=widgets.Layout(width='520px'),
        style={'description_width': '130px'},
    )
    scope = widgets.Dropdown(
        options=['Italy only', 'Italy + EU banks'],
        value=USER_CONFIG.get('universe_scope', 'Italy only') if USER_CONFIG.get('universe_scope', 'Italy only') in ['Italy only', 'Italy + EU banks'] else 'Italy only',
        description='Scope',
        layout=widgets.Layout(width='520px'),
        style={'description_width': '130px'},
    )
    target = widgets.Text(value=str(USER_CONFIG.get('target_ticker', 'ISP.MI')), description='Target', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    selected = widgets.Textarea(
        value=', '.join(USER_CONFIG.get('selected_tickers', [])),
        description='Tickers',
        layout=widgets.Layout(width='720px', height='90px'),
        style={'description_width': '130px'},
    )
    force_refresh = widgets.Checkbox(value=bool(USER_CONFIG.get('force_refresh', False)), description='Force datacenter refresh', indent=False)
    api_fallbacks = widgets.Checkbox(value=bool(USER_CONFIG.get('use_api_price_fallbacks', True)), description='Use API/yfinance price fallbacks', indent=False)
    start_date = widgets.Text(value=str(USER_CONFIG.get('start_date', '2015-01-01')), description='Start date', layout=widgets.Layout(width='360px'), style={'description_width': '120px'})
    train_end = widgets.Text(value=str(USER_CONFIG.get('train_end', '2021-12-31')), description='Train end', layout=widgets.Layout(width='360px'), style={'description_width': '120px'})
    test_start = widgets.Text(value=str(USER_CONFIG.get('test_start', '2022-01-31')), description='Test start', layout=widgets.Layout(width='360px'), style={'description_width': '120px'})
    primary_model = widgets.Dropdown(options=['OLS', 'LASSO', 'Random Forest', 'Gradient Boosting'], value=USER_CONFIG.get('primary_model', 'Random Forest'), description='Primary model', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    quantiles = widgets.IntSlider(value=int(USER_CONFIG.get('n_quantiles', 5)), min=3, max=10, step=1, description='Quantiles', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    ensemble = widgets.Checkbox(value=bool(USER_CONFIG.get('use_ensemble_mispricing', True)), description='Use ensemble mispricing', indent=False)
    rf_trees = widgets.IntSlider(value=int(USER_CONFIG.get('rf_params', {}).get('n_estimators', 160)), min=50, max=500, step=10, description='RF trees', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    rf_leaf = widgets.IntSlider(value=int(USER_CONFIG.get('rf_params', {}).get('min_samples_leaf', 2)), min=1, max=10, step=1, description='RF leaf', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    cet1 = widgets.FloatSlider(value=float(USER_CONFIG.get('cet1_min', 11.0)), min=6.0, max=18.0, step=0.25, description='CET1 min', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    npl = widgets.FloatSlider(value=float(USER_CONFIG.get('npl_max', 0.08)), min=0.0, max=0.30, step=0.005, readout_format='.3f', description='NPL max', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})
    ldr = widgets.FloatSlider(value=float(USER_CONFIG.get('liq_min', 0.0)), min=0.0, max=1.5, step=0.05, description='Liq/LDR min', layout=widgets.Layout(width='520px'), style={'description_width': '130px'})

    tabs = widgets.Tab(children=[
        widgets.VBox([universe_name, scope, target, selected]),
        widgets.VBox([force_refresh, api_fallbacks, start_date, train_end, test_start]),
        widgets.VBox([primary_model, quantiles, ensemble, rf_trees, rf_leaf]),
        widgets.VBox([cet1, npl, ldr]),
    ])
    for idx, title in enumerate(['Universe & Peers', 'Data', 'Models', 'Screening']):
        tabs.set_title(idx, title)

    out = widgets.Output()

    def apply_control_center_config(_=None):
        with out:
            out.clear_output(wait=True)
            USER_CONFIG['universe_name'] = universe_name.value
            USER_CONFIG['universe_scope'] = scope.value
            USER_CONFIG['target_ticker'] = target.value.strip().upper()
            USER_CONFIG['selected_tickers'] = _parse_tickers(selected.value) if '_parse_tickers' in globals() else [x.strip().upper() for x in selected.value.split(',') if x.strip()]
            USER_CONFIG['force_refresh'] = bool(force_refresh.value)
            USER_CONFIG['use_api_price_fallbacks'] = bool(api_fallbacks.value)
            USER_CONFIG['start_date'] = start_date.value.strip()
            USER_CONFIG['train_end'] = train_end.value.strip()
            USER_CONFIG['test_start'] = test_start.value.strip()
            USER_CONFIG['primary_model'] = primary_model.value
            USER_CONFIG['n_quantiles'] = int(quantiles.value)
            USER_CONFIG['use_ensemble_mispricing'] = bool(ensemble.value)
            USER_CONFIG.setdefault('rf_params', {})['n_estimators'] = int(rf_trees.value)
            USER_CONFIG.setdefault('rf_params', {})['min_samples_leaf'] = int(rf_leaf.value)
            USER_CONFIG['cet1_min'] = float(cet1.value)
            USER_CONFIG['npl_max'] = float(npl.value)
            USER_CONFIG['liq_min'] = float(ldr.value)
            if '_sync_experiment_globals' in globals():
                _sync_experiment_globals()
            print('Configurazione applicata. Esegui le celle successive per rigenerare datacenter e modelli.')
            debug_experiment_config(USER_CONFIG)

    apply_button = widgets.Button(description='Applica configurazione esperimento', button_style='success', icon='check')
    apply_button.on_click(apply_control_center_config)

    display(HTML("<h3>Italian Banks ML Lab · Control Center</h3>"))
    display(widgets.VBox([tabs, apply_button, out]))
    debug_experiment_config(USER_CONFIG)
else:
    display(HTML('<h3>Italian Banks ML Lab · Control Center</h3>'))
    display(HTML(f"<p><b>ipywidgets non disponibile.</b> Usa USER_CONFIG direttamente. Errore: {WIDGETS_IMPORT_ERROR}</p>"))
    debug_experiment_config(USER_CONFIG)


## 3. Panel scelta aziende

Costruiamo un pannello di selezione usando i soliti dataset/cache della piattaforma: universe master, screener, Finviz export e pannelli ML gia' presenti. Se `ipywidgets` e' disponibile, appare un selettore interattivo; altrimenti resta disponibile il DataFrame `company_panel` e la lista `selected_tickers`.

In [3]:
company_panel = build_company_selection_panel(
    project_root=PROJECT_ROOT,
    default_selected=USER_CONFIG.get('selected_tickers', DEFAULT_SELECTED_TICKERS),
)

italian_banks_panel = build_italian_banks_universe(company_panel)
if not italian_banks_panel.empty:
    company_panel = company_panel.copy()
    company_panel['universe_flag'] = ''
    company_panel.loc[company_panel['ticker'].astype(str).str.upper().isin(italian_banks_panel['ticker'].astype(str).str.upper()), 'universe_flag'] = 'italian_banks_extended'
    default_universe_tickers = italian_banks_panel['ticker'].dropna().astype(str).str.upper().drop_duplicates().tolist()
    if USER_CONFIG.get('universe_scope') == 'Italy only' and default_universe_tickers:
        USER_CONFIG['selected_tickers'] = [t for t in default_universe_tickers if t in set(DEFAULT_SELECTED_TICKERS + WEALTH_BANK_TICKERS)] or default_universe_tickers
else:
    default_universe_tickers = USER_CONFIG.get('selected_tickers', DEFAULT_SELECTED_TICKERS)

if company_panel.empty:
    selected_tickers = USER_CONFIG.get('selected_tickers', DEFAULT_SELECTED_TICKERS).copy()
    print('Company selection panel vuoto: uso tickers da USER_CONFIG.')
else:
    selected_tickers = USER_CONFIG.get('selected_tickers') or default_universe_tickers or DEFAULT_SELECTED_TICKERS.copy()
    display_cols = [
        'selected', 'universe_flag', 'ticker', 'company_name', 'country', 'sector', 'industry',
        'index_membership', 'screener_score', 'source_count', 'data_sources',
    ]
    display(company_panel[[col for col in display_cols if col in company_panel.columns]].head(80))

try:
    company_selector = build_company_selection_widget(
        company_panel,
        default_tickers=selected_tickers,
        max_options=700,
    )
    company_selector['display']()
    print("Default: universo banche italiane allargato. Puoi aggiungere altri ticker nel widget se vuoi testare peer europei o financial comparables.")
except ImportError:
    company_selector = None
    print('ipywidgets non disponibile: modifica USER_CONFIG["selected_tickers"] manualmente se vuoi cambiare universo.')

selected_tickers


,selected,ticker,company_name,country,sector,industry,index_membership,screener_score,source_count,data_sources
0,True,BAMI.MI,Banco BPM S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
1,True,BMED.MI,Banca Mediolanum S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
2,True,BMPS.MI,Banca Monte dei Paschi di Siena S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
3,True,BPE.MI,BPER Banca SpA,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
4,True,FBK.MI,FinecoBank Banca Fineco S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
5,True,ISP.MI,Intesa Sanpaolo S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
6,True,MB.MI,Mediobanca Banca di Credito Finanziario S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
7,True,UCG.MI,UniCredit S.p.A.,IT,Banks,<NA>,static_ftse_mib,0.0,7,"FinvizLatestCrossSection.csv, FinvizSecurities..."
8,True,BGN.MI,Banca Generali S.p.A.,IT,Banks,<NA>,NaN,NaN,1,italian_banks_panel.csv
9,True,CE.MI,Credito Emiliano S.p.A.,IT,Banks,<NA>,NaN,NaN,1,italian_banks_panel.csv


Puoi raffinare la selezione qui. La prossima cella userà i ticker selezionati nel widget + USER_CONFIG.


['ISP.MI',
 'UCG.MI',
 'BAMI.MI',
 'BPE.MI',
 'BMPS.MI',
 'CE.MI',
 'BGN.MI',
 'FBK.MI',
 'BMED.MI',
 'MB.MI']

## 4. Caricamento & preprocessing dati

### Banking datacenter

In questa sezione il notebook costruisce un piccolo **datacenter bancario**: un set coerente di tabelle che alimentano tutto il lab. Le tabelle principali sono `banks_universe` (anagrafica e classificazione banche), `banks_macro_regulatory` (macro/regolamentare da ECB/Banca d'Italia quando configurato), `banks_fundamentals_panel` (fondamentali per banca/data) e `italian_banks_panel` (panel `date × ticker` usato dai modelli ML).

Il flag `force_refresh` forza il riallineamento con pipeline/API e rigenera i file anche se esistono già in cache. In modalità normale il notebook usa cache e CSV locali prima, poi provider API e infine `yfinance` come fallback.

### Pulizia prezzi e BMPS

Alcuni ticker bancari, in particolare BMPS, possono avere serie storiche con reverse split, cambi di scala e punti prezzo fuori scala. Il lab ora lavora con `mkt_price_clean` e `mkt_market_cap_clean`: i prezzi vengono controllati per outlier per ticker su scala logaritmica e winsorizzati in modo conservativo. I modelli usano market cap pulita e `log(market cap)` per ridurre l'impatto del prezzo grezzo sui grafici e sulle regressioni.

### Fondamentali alternativi

Quando CET1, NPL, LDR o ROA/ROE mancano, il notebook calcola proxy espliciti: value/quality da P/B, P/E, momentum, leverage/profitability quando disponibili. Queste colonne `_alt` non fingono copertura regolamentare completa: servono a non perdere completamente banche meno coperte e restano tracciabili nel report finale.


In [ ]:
# ============================================================
# Banking datacenter + API fallback + robust preprocessing
# ============================================================

import time
import requests

BANKS_PIPELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BANKS_PIPELINE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RAW_PRICE_COL = 'mkt_price'
RAW_MARKET_CAP_COL = 'mkt_market_cap'
CLEAN_PRICE_COL = 'mkt_price_clean'
CLEAN_MARKET_CAP_COL = 'mkt_market_cap_clean'
DATA_FRESHNESS_HOURS = 24 * 7


def _file_is_fresh(path: Path, max_age_hours: int = DATA_FRESHNESS_HOURS) -> bool:
    if not Path(path).exists() or Path(path).stat().st_size <= 0:
        return False
    age = (pd.Timestamp.utcnow() - pd.Timestamp(Path(path).stat().st_mtime, unit='s', tz='UTC')).total_seconds() / 3600
    return age <= max_age_hours


def _read_csv_if_exists(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path) if Path(path).exists() and Path(path).stat().st_size > 0 else pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def get_api_key(name: str) -> str | None:
    """Recupera una API key da google.colab.userdata oppure da env vars locali."""
    candidates = [name, name.upper(), name.lower(), name.replace('.', '_'), name.replace('.', '_').upper()]
    try:
        from google.colab import userdata
        for key in candidates:
            try:
                value = userdata.get(key)
                if value:
                    return str(value).strip()
            except Exception:
                pass
    except Exception:
        pass
    for key in candidates:
        value = os.environ.get(key)
        if value:
            return str(value).strip()
    return None


def _standardize_price_frame(df: pd.DataFrame, ticker: str, source: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out.columns = [str(c).strip().lower().replace(' ', '_') for c in out.columns]
    rename = {
        'datetime': 'date', 'timestamp': 'date', 'time': 'date',
        'adjusted_close': 'adjusted_close', 'adj_close': 'adjusted_close', 'adjustedclose': 'adjusted_close',
        'close': 'close', 'volume': 'volume',
    }
    out = out.rename(columns={k: v for k, v in rename.items() if k in out.columns})
    if 'date' not in out.columns:
        return pd.DataFrame()
    if 'close' not in out.columns and 'adjusted_close' in out.columns:
        out['close'] = out['adjusted_close']
    if 'adjusted_close' not in out.columns and 'close' in out.columns:
        out['adjusted_close'] = out['close']
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out['ticker'] = ticker.upper()
    out['source'] = source
    for col in ['close', 'adjusted_close', 'volume']:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors='coerce')
    return out[['date', 'ticker', 'close', 'adjusted_close', 'volume', 'source']].dropna(subset=['date', 'close'])


def fetch_prices_eodhd(ticker, start, end=None):
    key = get_api_key('EODHD_API_KEY') or get_api_key('EODHD')
    if not key:
        print(f'EODHD key missing for {ticker}')
        return pd.DataFrame()
    symbol = ticker.replace('.MI', '.MI')
    url = f'https://eodhd.com/api/eod/{symbol}'
    params = {'api_token': key, 'fmt': 'json', 'from': start}
    if end:
        params['to'] = end
    try:
        r = requests.get(url, params=params, timeout=25)
        if r.status_code == 429:
            time.sleep(1.5)
            r = requests.get(url, params=params, timeout=25)
        r.raise_for_status()
        return _standardize_price_frame(pd.DataFrame(r.json()), ticker, 'eodhd_api')
    except Exception as exc:
        print(f'EODHD failed {ticker}: {exc}')
        return pd.DataFrame()


def fetch_prices_finnhub(ticker, start, end=None):
    key = get_api_key('FINNHUB_API_KEY') or get_api_key('FINNHUB')
    if not key:
        print(f'Finnhub key missing for {ticker}')
        return pd.DataFrame()
    start_ts = int(pd.Timestamp(start).timestamp())
    end_ts = int(pd.Timestamp(end or pd.Timestamp.today()).timestamp())
    url = 'https://finnhub.io/api/v1/stock/candle'
    params = {'symbol': ticker, 'resolution': 'D', 'from': start_ts, 'to': end_ts, 'token': key}
    try:
        r = requests.get(url, params=params, timeout=25)
        if r.status_code == 429:
            time.sleep(1.5)
            r = requests.get(url, params=params, timeout=25)
        r.raise_for_status()
        js = r.json()
        if js.get('s') != 'ok':
            return pd.DataFrame()
        dfp = pd.DataFrame({'date': pd.to_datetime(js['t'], unit='s'), 'close': js['c'], 'adjusted_close': js['c'], 'volume': js.get('v', np.nan)})
        return _standardize_price_frame(dfp, ticker, 'finnhub_api')
    except Exception as exc:
        print(f'Finnhub failed {ticker}: {exc}')
        return pd.DataFrame()


def fetch_prices_alpha_vantage(ticker, start, end=None):
    key = get_api_key('ALPHA_VANTAGE_API_KEY') or get_api_key('alpha_vantage_api_key')
    if not key:
        print(f'AlphaVantage key missing for {ticker}')
        return pd.DataFrame()
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'TIME_SERIES_DAILY_ADJUSTED', 'symbol': ticker, 'outputsize': 'full', 'apikey': key}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json().get('Time Series (Daily)', {})
        if not data:
            return pd.DataFrame()
        dfp = pd.DataFrame.from_dict(data, orient='index').reset_index().rename(columns={'index': 'date', '4. close': 'close', '5. adjusted close': 'adjusted_close', '6. volume': 'volume'})
        dfp = _standardize_price_frame(dfp, ticker, 'alpha_vantage_api')
        return dfp[dfp['date'] >= pd.Timestamp(start)]
    except Exception as exc:
        print(f'AlphaVantage failed {ticker}: {exc}')
        return pd.DataFrame()


def fetch_prices_tiingo(ticker, start, end=None):
    key = get_api_key('TIINGO_API_KEY') or get_api_key('TIINGO.API') or get_api_key('TIINGO')
    if not key:
        print(f'Tiingo key missing for {ticker}')
        return pd.DataFrame()
    url = f'https://api.tiingo.com/tiingo/daily/{ticker}/prices'
    params = {'startDate': start, 'token': key}
    if end:
        params['endDate'] = end
    try:
        r = requests.get(url, params=params, timeout=25)
        r.raise_for_status()
        dfp = pd.DataFrame(r.json()).rename(columns={'adjClose': 'adjusted_close'})
        return _standardize_price_frame(dfp, ticker, 'tiingo_api')
    except Exception as exc:
        print(f'Tiingo failed {ticker}: {exc}')
        return pd.DataFrame()


def fetch_prices_polygon(ticker, start, end=None):
    key = get_api_key('POLYGON_API_KEY') or get_api_key('polygon_api_key')
    if not key:
        print(f'Polygon key missing for {ticker}')
        return pd.DataFrame()
    end = end or pd.Timestamp.today().date().isoformat()
    url = f'https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}'
    params = {'adjusted': 'true', 'sort': 'asc', 'limit': 50000, 'apiKey': key}
    try:
        r = requests.get(url, params=params, timeout=25)
        r.raise_for_status()
        rows = r.json().get('results', [])
        if not rows:
            return pd.DataFrame()
        dfp = pd.DataFrame(rows)
        dfp = pd.DataFrame({'date': pd.to_datetime(dfp['t'], unit='ms'), 'close': dfp['c'], 'adjusted_close': dfp['c'], 'volume': dfp.get('v', np.nan)})
        return _standardize_price_frame(dfp, ticker, 'polygon_api')
    except Exception as exc:
        print(f'Polygon failed {ticker}: {exc}')
        return pd.DataFrame()


def fetch_prices_yfinance(ticker, start, end=None):
    try:
        import yfinance as yf
        raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False, threads=False)
        if raw.empty:
            return pd.DataFrame()
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        dfp = raw.reset_index().rename(columns={'Date': 'date', 'Close': 'close', 'Adj Close': 'adjusted_close', 'Volume': 'volume'})
        return _standardize_price_frame(dfp, ticker, 'yfinance_fallback')
    except Exception as exc:
        print(f'yfinance failed {ticker}: {exc}')
        return pd.DataFrame()


def fetch_prices_with_fallbacks(ticker: str, start: str, end: str | None = None) -> pd.DataFrame:
    """Try EODHD, Finnhub, AlphaVantage, Tiingo, Polygon, then yfinance."""
    providers = {
        'eodhd': fetch_prices_eodhd,
        'finnhub': fetch_prices_finnhub,
        'alpha_vantage': fetch_prices_alpha_vantage,
        'tiingo': fetch_prices_tiingo,
        'polygon': fetch_prices_polygon,
        'yfinance': fetch_prices_yfinance,
    }
    for provider_name in USER_CONFIG.get('api_provider_order', list(providers)):
        fn = providers.get(provider_name)
        if not fn:
            continue
        dfp = fn(ticker, start, end)
        if dfp is not None and not dfp.empty:
            return dfp
    return pd.DataFrame()


def build_provider_price_panel(tickers, start='2015-01-01', end=None, save_path=None):
    frames = []
    for ticker in _parse_tickers(tickers):
        daily = fetch_prices_with_fallbacks(ticker, start, end)
        if daily.empty:
            continue
        monthly = daily.set_index('date').resample('ME').agg({'adjusted_close': 'last', 'volume': 'sum', 'source': 'last'}).dropna(subset=['adjusted_close']).reset_index()
        monthly['ticker'] = ticker
        monthly[RAW_PRICE_COL] = monthly['adjusted_close']
        monthly[RAW_MARKET_CAP_COL] = monthly[RAW_PRICE_COL]
        monthly['target_source'] = monthly['source'].astype(str) + '_price_proxy'
        monthly['data_source'] = monthly['source']
        monthly['country'] = 'IT'
        monthly['sector'] = 'Banks'
        frames.append(monthly[['date', 'ticker', RAW_PRICE_COL, RAW_MARKET_CAP_COL, 'volume', 'target_source', 'data_source', 'country', 'sector']])
        time.sleep(0.25)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True, sort=False).sort_values(['ticker', 'date'])
    out['ret_1m'] = out.groupby('ticker')[RAW_PRICE_COL].pct_change()
    out['mkt_vol_1y'] = out.groupby('ticker')['ret_1m'].rolling(12, min_periods=4).std().reset_index(level=0, drop=True)
    out['mkt_mom_6m'] = out.groupby('ticker')[RAW_PRICE_COL].pct_change(6)
    out['mkt_mom_12m'] = out.groupby('ticker')[RAW_PRICE_COL].pct_change(12)
    roll_max = out.groupby('ticker')[RAW_PRICE_COL].rolling(12, min_periods=4).max().reset_index(level=0, drop=True)
    out['mkt_drawdown_1y'] = out[RAW_PRICE_COL] / roll_max - 1
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        out.to_csv(save_path, index=False)
    return out.reset_index(drop=True)


def resolve_data_path(data_path=None, candidates=None):
    paths = []
    if data_path is not None:
        paths.append(Path(data_path).expanduser())
    paths.extend(Path(path).expanduser() for path in (candidates or []))
    for path in paths:
        if path.exists() and path.stat().st_size > 0:
            return path
    print('CSV panel non trovato nei percorsi standard. Percorsi controllati:')
    for path in paths:
        print(' -', path)
    return None


def filter_universe(df: pd.DataFrame, universe_filter: dict | None = None) -> pd.DataFrame:
    """Notebook-safe fallback filter for country/sector universe selection."""
    if df is None or df.empty or not universe_filter:
        return df.copy() if isinstance(df, pd.DataFrame) else pd.DataFrame()
    out = df.copy()
    for col, wanted in universe_filter.items():
        if col not in out.columns or wanted is None:
            continue
        wanted_values = wanted if isinstance(wanted, (list, tuple, set)) else [wanted]
        wanted_norm = {str(v).strip().upper() for v in wanted_values}
        series = out[col].astype(str).str.strip().str.upper()
        if col.lower() == 'country' and wanted_norm & {'IT', 'ITALY', 'ITALIA'}:
            mask = series.isin({'IT', 'ITALY', 'ITALIA'})
        elif col.lower() == 'sector' and 'BANKS' in wanted_norm:
            mask = series.eq('BANKS') | series.str.contains('BANK', na=False)
        else:
            mask = series.isin(wanted_norm)
        out = out[mask].copy()
    return out.reset_index(drop=True)


def normalize_price_panel(df: pd.DataFrame,
                          price_col: str = RAW_PRICE_COL,
                          mcap_col: str = RAW_MARKET_CAP_COL,
                          max_log_price_z: float = 4.0) -> pd.DataFrame:
    """
    Identifica e tratta prezzi anomali, inclusi reverse split tipo BMPS.
    Usa log-price per ticker, z-score temporale robusto e winsorizzazione su log-prezzi.
    Ritorna price_outlier_flag, mkt_price_clean, mkt_market_cap_clean.
    """
    out = df.copy()
    if price_col not in out.columns:
        out[price_col] = np.nan
    if mcap_col not in out.columns:
        out[mcap_col] = out[price_col]
    out[price_col] = pd.to_numeric(out[price_col], errors='coerce')
    out[mcap_col] = pd.to_numeric(out[mcap_col], errors='coerce')
    out['price_outlier_flag'] = False
    out['log_price_raw'] = np.log(out[price_col].where(out[price_col] > 0))
    out['log_price_clean'] = out['log_price_raw']

    for ticker, idx in out.groupby('ticker').groups.items():
        lp = pd.to_numeric(out.loc[idx, 'log_price_raw'], errors='coerce')
        valid = lp.dropna()
        if valid.shape[0] < 8:
            continue
        med = valid.median()
        mad = (valid - med).abs().median()
        scale = 1.4826 * mad if mad and mad > 0 else valid.std(ddof=0)
        if not scale or pd.isna(scale) or scale == 0:
            continue
        z = (lp - med) / scale
        flags = z.abs() > max_log_price_z
        # BMPS often has structural scale breaks: use slightly more conservative flagging.
        if str(ticker).upper().startswith('BMPS'):
            flags = flags | (z.abs() > min(max_log_price_z, 3.25))
        lower = med - max_log_price_z * scale
        upper = med + max_log_price_z * scale
        clean_log = lp.clip(lower=lower, upper=upper)
        out.loc[idx, 'price_outlier_flag'] = flags.fillna(False).values
        out.loc[idx, 'log_price_clean'] = clean_log

    out[CLEAN_PRICE_COL] = np.exp(out['log_price_clean'])
    ratio = out[CLEAN_PRICE_COL] / out[price_col].replace(0, np.nan)
    out[CLEAN_MARKET_CAP_COL] = out[mcap_col] * ratio.fillna(1.0)
    out['price_clean_method'] = np.where(out['price_outlier_flag'], 'log_zscore_winsorized', 'raw')
    return out


def plot_price_cleaning_diagnostics(df: pd.DataFrame, ticker: str = 'BMPS.MI') -> None:
    """Plot raw vs clean log prices and outlier flags for one ticker."""
    if not PLOTLY_AVAILABLE or df is None or df.empty or 'ticker' not in df.columns:
        return
    g = df[df['ticker'].astype(str).str.upper().eq(ticker.upper())].copy()
    if g.empty or 'log_price_raw' not in g.columns or 'log_price_clean' not in g.columns:
        return
    plot = g[['date', 'log_price_raw', 'log_price_clean', 'price_outlier_flag']].melt(
        id_vars=['date', 'price_outlier_flag'], var_name='series', value_name='log_price'
    )
    fig = px.line(plot, x='date', y='log_price', color='series', title=f'{ticker} · raw vs clean log price')
    flagged = g[g['price_outlier_flag'].fillna(False)]
    if not flagged.empty:
        fig.add_scatter(x=flagged['date'], y=flagged['log_price_raw'], mode='markers', name='outlier flag', marker=dict(color='#da7101', size=9, symbol='x'))
    fig.update_layout(template='plotly_white', height=460)
    fig.show()


def build_data_coverage_log(panel: pd.DataFrame, train_end: str = TRAIN_END, test_start: str = TEST_START) -> pd.DataFrame:
    """Coverage diagnostics for core fundamentals and effective train/test universe size."""
    rows = []
    core = ['fund_roa_lag', 'fund_roe_lag', 'fund_cet1_lag', 'fund_npl_ratio_lag', 'fund_ldr_lag']
    for col in core:
        rows.append({'metric': f'coverage_{col}', 'value': float(panel[col].notna().mean()) if col in panel.columns and len(panel) else np.nan})
    if 'fundamental_coverage_flag' in panel.columns and len(panel):
        rows.append({'metric': 'alternative_fundamentals_share', 'value': float(panel['fundamental_coverage_flag'].astype(str).eq('alternative_proxy').mean())})
    train = panel[pd.to_datetime(panel['date'], errors='coerce') <= pd.Timestamp(train_end)] if 'date' in panel else pd.DataFrame()
    test = panel[pd.to_datetime(panel['date'], errors='coerce') >= pd.Timestamp(test_start)] if 'date' in panel else pd.DataFrame()
    rows.extend([
        {'metric': 'n_banks_train', 'value': int(train['ticker'].nunique()) if 'ticker' in train else 0},
        {'metric': 'n_banks_test', 'value': int(test['ticker'].nunique()) if 'ticker' in test else 0},
        {'metric': 'n_obs_train', 'value': int(len(train))},
        {'metric': 'n_obs_test', 'value': int(len(test))},
        {'metric': 'price_outlier_share', 'value': float(panel.get('price_outlier_flag', pd.Series(False, index=panel.index)).mean()) if len(panel) else np.nan},
    ])
    return pd.DataFrame(rows)


def augment_fundamentals_with_alternatives(df: pd.DataFrame) -> pd.DataFrame:
    """
    Completa e arricchisce il pannello dei fondamentali bancari.
    TODO data ingestion: popolare da CSV regolamentari/manuali le colonne annuali o trimestrali
    CET1/NPL/LCR/NSFR/BTP exposure quando disponibili dalle banche, ECB o Banca d'Italia.
    """
    out = df.copy()
    required_bank_cols = [
        'fund_cet1', 'fund_tier1', 'fund_total_capital', 'fund_leverage_ratio',
        'fund_npl_gross', 'fund_npl_net', 'fund_npl_ratio', 'fund_cost_of_risk', 'fund_coverage_ratio',
        'fund_ldr', 'fund_lcr', 'fund_nsfr',
        'fund_btp_exposure', 'fund_btp_assets_ratio', 'fund_securities_duration',
        'fund_nopat', 'fund_wacc', 'fund_invested_capital', 'fund_eva',
        'fund_roa', 'fund_roe', 'fund_assets',
    ]
    raw_proxy_cols = [
        'total_assets', 'equity', 'net_income', 'nopat', 'invested_capital', 'wacc',
        'loans', 'loans_to_customers', 'customer_deposits', 'deposits',
        'loan_loss_provisions', 'npl_gross', 'npl_net', 'btp_exposure',
        'mkt_pb', 'mkt_pe', 'mkt_mom_12m', 'dy', 'dividend_yield',
    ]
    for col in required_bank_cols + raw_proxy_cols:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors='coerce')

    total_assets = out['total_assets'].fillna(out['fund_assets'])
    equity = out['equity']
    net_income = out['net_income']
    loans = out['loans_to_customers'].fillna(out['loans'])
    deposits = out['customer_deposits'].fillna(out['deposits'])

    out['profitability_alt'] = net_income / total_assets.replace(0, np.nan)
    out['roe_alt'] = net_income / equity.replace(0, np.nan)
    out['leverage_alt'] = total_assets / equity.replace(0, np.nan)
    out['capital_proxy_alt'] = 1 / out['leverage_alt'].replace(0, np.nan)
    out['credit_risk_alt'] = out['loan_loss_provisions'] / loans.replace(0, np.nan)
    out['fund_ldr_alt'] = loans / deposits.replace(0, np.nan)
    out['fund_coverage_ratio_alt'] = (out['npl_gross'] - out['npl_net']) / out['npl_gross'].replace(0, np.nan)
    out['fund_btp_assets_ratio_alt'] = out['btp_exposure'] / total_assets.replace(0, np.nan)
    out['fund_eva_alt'] = out['nopat'].fillna(out['fund_nopat']) - out['wacc'].fillna(out['fund_wacc']) * out['invested_capital'].fillna(out['fund_invested_capital'])

    out['fund_roa'] = out['fund_roa'].fillna(out['profitability_alt'])
    out['fund_roe'] = out['fund_roe'].fillna(out['roe_alt'])
    out['fund_cet1'] = out['fund_cet1'].fillna(out['capital_proxy_alt'])
    out['fund_leverage_ratio'] = out['fund_leverage_ratio'].fillna(out['capital_proxy_alt'])
    out['fund_npl_ratio'] = out['fund_npl_ratio'].fillna(out['credit_risk_alt'])
    out['fund_ldr'] = out['fund_ldr'].fillna(out['fund_ldr_alt'])
    out['fund_coverage_ratio'] = out['fund_coverage_ratio'].fillna(out['fund_coverage_ratio_alt'])
    out['fund_btp_assets_ratio'] = out['fund_btp_assets_ratio'].fillna(out['fund_btp_assets_ratio_alt'])
    out['fund_eva'] = out['fund_eva'].fillna(out['fund_eva_alt'])

    value = -pd.to_numeric(out.get('mkt_pb', np.nan), errors='coerce').rank(pct=True)
    pe_quality = -pd.to_numeric(out.get('mkt_pe', np.nan), errors='coerce').rank(pct=True)
    momentum = pd.to_numeric(out.get('mkt_mom_12m', np.nan), errors='coerce').rank(pct=True)
    dy = pd.to_numeric(out.get('dy', out.get('dividend_yield', np.nan)), errors='coerce').rank(pct=True)
    out['value_quality_alt'] = pd.concat([value, pe_quality, momentum, dy], axis=1).mean(axis=1)

    classic_cols = ['fund_roa', 'fund_roe', 'fund_cet1', 'fund_npl_ratio', 'fund_ldr']
    out['fundamental_full_flag'] = out[classic_cols].notna().mean(axis=1).ge(0.8)
    out['fundamental_coverage_flag'] = np.where(out['fundamental_full_flag'], 'classic_or_full', 'alternative_proxy')
    return out


def build_experiment_yfinance_panel(tickers, start='2015-01-01', end=None, save_path=None):
    if USER_CONFIG.get('use_api_price_fallbacks', True):
        api_panel = build_provider_price_panel(tickers, start=start, end=end, save_path=save_path)
        if not api_panel.empty:
            return api_panel
    if BANKING_DATA_ENGINE_AVAILABLE and core_build_yfinance_bank_panel is not None:
        return core_build_yfinance_bank_panel(
            tickers=_parse_tickers(tickers),
            start=start,
            end=end,
            price_col_name=RAW_PRICE_COL,
            market_cap_col_name=RAW_MARKET_CAP_COL,
            save_path=save_path,
        )
    raise ImportError('Banking data engine non disponibile: impossibile costruire fallback prezzi condiviso.')


def prepare_experiment_panel(df: pd.DataFrame) -> pd.DataFrame:
    global PRICE_COL, MARKET_CAP_COL
    out = augment_fundamentals_with_alternatives(df.copy())
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out['ticker'] = out['ticker'].astype(str).str.upper()
    if RAW_MARKET_CAP_COL not in out.columns and 'market_value' in out.columns:
        out[RAW_MARKET_CAP_COL] = pd.to_numeric(out['market_value'], errors='coerce')
    if RAW_PRICE_COL not in out.columns:
        for candidate in ['price', 'close', 'adj_close', 'last_price']:
            if candidate in out.columns:
                out[RAW_PRICE_COL] = pd.to_numeric(out[candidate], errors='coerce')
                break
    if RAW_MARKET_CAP_COL not in out.columns and RAW_PRICE_COL in out.columns:
        out[RAW_MARKET_CAP_COL] = pd.to_numeric(out[RAW_PRICE_COL], errors='coerce')
        out['target_source'] = out.get('target_source', 'price_proxy')
    out = normalize_price_panel(out, price_col=RAW_PRICE_COL, cap_col=RAW_MARKET_CAP_COL)
    PRICE_COL = CLEAN_PRICE_COL
    MARKET_CAP_COL = CLEAN_MARKET_CAP_COL
    if 'bank_id' not in out.columns and not banks_universe.empty and 'ticker' in banks_universe.columns:
        ticker_map = banks_universe[['bank_id', 'ticker', 'status', 'group_name', 'bank_category']].dropna(subset=['ticker']).copy()
        ticker_map['ticker'] = ticker_map['ticker'].astype(str).str.upper()
        out = out.merge(ticker_map.drop_duplicates('ticker'), on='ticker', how='left')
    out = fundamentals.lag_fundamentals(out, lag_months=3)
    out = fundamentals.add_log_mcap_target(out, mkt_cap_col=MARKET_CAP_COL, out_col=LOG_MCAP_COL)
    out = fundamentals.add_forward_returns(out, price_col=PRICE_COL, horizon_months=1, out_col=FORWARD_RETURN_COL)
    out['date'] = pd.to_datetime(out['date'])
    return out.sort_values(['date', 'ticker']).reset_index(drop=True)


def refresh_banking_datacenter(force_refresh: bool = False) -> dict:
    """Riallinea l'intero datacenter bancario usato nel lab."""
    selected = _parse_tickers(USER_CONFIG.get('selected_tickers', DEFAULT_SELECTED_TICKERS))
    paths = {
        'banks_universe': BANKS_PIPELINE_OUTPUT_DIR / 'banks_universe.csv',
        'banks_macro_regulatory': BANKS_PIPELINE_OUTPUT_DIR / 'banks_macro_regulatory.csv',
        'banks_fundamentals_panel': BANKS_PIPELINE_OUTPUT_DIR / 'banks_fundamentals_panel.csv',
        'italian_banks_panel': PROJECT_DATA_DIR / 'italian_banks_panel.csv',
    }
    needs_refresh = force_refresh or any(not _file_is_fresh(path) for path in paths.values() if path.name != 'italian_banks_panel.csv')
    artifacts = {}
    if BANKING_DATA_ENGINE_AVAILABLE and (needs_refresh or USER_CONFIG.get('refresh_banks_universe', True)):
        try:
            artifacts = run_banks_data_pipeline(
                output_dir=BANKS_PIPELINE_OUTPUT_DIR,
                cache_dir=BANKS_PIPELINE_CACHE_DIR,
                bds_exports=USER_CONFIG.get('bds_exports', {}),
                external_fundamentals_path=USER_CONFIG.get('external_fundamentals_path'),
                market_start=USER_CONFIG.get('start_date', '2015-01-01'),
                include_market=False,
                include_ecb_macro=bool(USER_CONFIG.get('include_ecb_macro', False)),
            )
        except Exception as exc:
            print(f'Banking datacenter universe refresh saltato: {exc}')
    banks_univ = artifacts.get('banks_universe', _read_csv_if_exists(paths['banks_universe']))
    macro = artifacts.get('banks_macro_regulatory', _read_csv_if_exists(paths['banks_macro_regulatory']))
    fundamentals_panel = artifacts.get('banks_fundamentals_panel', _read_csv_if_exists(paths['banks_fundamentals_panel']))
    if not fundamentals_panel.empty:
        fundamentals_panel = augment_fundamentals_with_alternatives(fundamentals_panel)
        fundamentals_panel.to_csv(paths['banks_fundamentals_panel'], index=False)

    data_path_resolved = None if USER_CONFIG.get('data_mode') == 'yfinance_refresh' else resolve_data_path(DATA_PATH, CANDIDATE_DATA_PATHS)
    if data_path_resolved is not None and not force_refresh and USER_CONFIG.get('data_mode') != 'yfinance_refresh':
        panel = load_panel_csv(data_path_resolved)
        source = str(data_path_resolved)
    elif USER_CONFIG.get('data_mode') == 'csv_only':
        raise FileNotFoundError('CSV richiesto ma non trovato. Cambia data_mode in auto/yfinance_refresh oppure imposta DATA_PATH.')
    else:
        save_path = PROJECT_DATA_DIR / 'italian_banks_panel.csv' if USER_CONFIG.get('save_generated_panel') else None
        panel = build_experiment_yfinance_panel(selected, start=USER_CONFIG.get('start_date', '2015-01-01'), end=USER_CONFIG.get('end_date'), save_path=save_path)
        source = 'api_or_yfinance_fallback'
    if panel.empty and USER_CONFIG.get('data_mode') == 'auto':
        save_path = PROJECT_DATA_DIR / 'italian_banks_panel.csv' if USER_CONFIG.get('save_generated_panel') else None
        panel = build_experiment_yfinance_panel(selected, start=USER_CONFIG.get('start_date', '2015-01-01'), end=USER_CONFIG.get('end_date'), save_path=save_path)
        source = 'api_or_yfinance_fallback_after_empty_filter'
    return {
        'banks_universe': banks_univ,
        'banks_macro_regulatory': macro,
        'banks_fundamentals_panel': fundamentals_panel,
        'italian_banks_panel': panel,
        'data_source_used': source,
    }


# Resolve selected tickers from widget + config.
if 'company_selector' in globals() and company_selector is not None:
    widget_tickers = company_selector['selected_tickers']()
    if widget_tickers:
        USER_CONFIG['selected_tickers'] = widget_tickers
elif 'selected_tickers' in globals() and selected_tickers:
    USER_CONFIG['selected_tickers'] = selected_tickers

selected_tickers = _parse_tickers(USER_CONFIG.get('selected_tickers', DEFAULT_SELECTED_TICKERS))
datacenter = refresh_banking_datacenter(force_refresh=USER_CONFIG.get('force_refresh', False))
banks_universe = datacenter['banks_universe']
banks_macro_regulatory = datacenter['banks_macro_regulatory']
banks_fundamentals_panel = datacenter['banks_fundamentals_panel']
italian_banks_panel = datacenter['italian_banks_panel']
DATA_SOURCE_USED = datacenter.get('data_source_used', 'unknown')

if not banks_universe.empty and {'ticker', 'country'}.issubset(banks_universe.columns):
    listed_it = banks_universe[(banks_universe['country'].astype(str).eq('IT')) & banks_universe['ticker'].notna()]['ticker'].astype(str).str.upper().unique().tolist()
    if listed_it:
        ITALIAN_BANK_UNIVERSES['Italian listed from banking universe'] = listed_it

df = filter_universe(italian_banks_panel, UNIVERSE_FILTER)
if selected_tickers:
    selected_ticker_set = set(selected_tickers)
    df = df[df['ticker'].astype(str).str.upper().isin(selected_ticker_set)].copy()
    print(f'Aziende selezionate nel dataset: {df["ticker"].nunique()} ticker, {len(df)} righe')
if df.empty and USER_CONFIG.get('data_mode') == 'auto':
    print('Il dataset trovato non contiene i ticker selezionati: passo al fallback API/yfinance.')
    save_path = PROJECT_DATA_DIR / 'italian_banks_panel.csv' if USER_CONFIG.get('save_generated_panel') else None
    df = build_experiment_yfinance_panel(selected_tickers, start=USER_CONFIG.get('start_date', '2015-01-01'), end=USER_CONFIG.get('end_date'), save_path=save_path)
    DATA_SOURCE_USED = 'api_or_yfinance_fallback_after_empty_filter'
    df = filter_universe(df, UNIVERSE_FILTER)
    if selected_tickers:
        df = df[df['ticker'].astype(str).str.upper().isin(set(selected_tickers))].copy()

df = prepare_experiment_panel(df)

ALT_FEATURES = [col for col in ['value_quality_alt', 'profitability_alt', 'leverage_alt', 'credit_risk_alt', 'sovereign_exposure_alt'] if col in df.columns]
for col in ALT_FEATURES:
    if col not in MARKET_FEATURES and col not in BANK_FUNDAMENTAL_FEATURES:
        MARKET_FEATURES.append(col)
FAIR_VALUE_FEATURES = BANK_FUNDAMENTAL_FEATURES + MARKET_FEATURES
available_fv = [col for col in FAIR_VALUE_FEATURES if col in df.columns and df[col].notna().mean() >= USER_CONFIG['min_feature_non_null_ratio']]
if not available_fv and USER_CONFIG.get('fallback_to_market_features'):
    available_fv = [col for col in MARKET_FEATURES if col in df.columns and df[col].notna().sum() >= 5]
FAIR_VALUE_FEATURES = available_fv
EXPECTED_RETURN_FEATURES = [col for col in (FAIR_VALUE_FEATURES + ['mispricing_z', 'mispricing_ens']) if col in df.columns or col in ['mispricing_z', 'mispricing_ens']]

required_cols = [LOG_MCAP_COL, FORWARD_RETURN_COL] + FAIR_VALUE_FEATURES
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    print('Colonne mancanti da controllare nel dataset:', missing_cols)
if not FAIR_VALUE_FEATURES:
    raise ValueError('Nessuna feature utilizzabile. Serve CSV con fondamentali/mercato o fallback API disponibili.')

df = fundamentals.filter_valid_rows(df, required_cols=[col for col in required_cols if col in df.columns])
df = df.sort_values(['date', 'ticker']).reset_index(drop=True)

DATA_SOURCE_SUMMARY = df.get('data_source', pd.Series(['unknown'] * len(df))).value_counts(normalize=True).rename('share').reset_index().rename(columns={'index': 'data_source'})
FUNDAMENTAL_COVERAGE_SUMMARY = df.get('fundamental_coverage_flag', pd.Series(['unknown'] * len(df))).value_counts(normalize=True).rename('share').reset_index().rename(columns={'index': 'coverage'})
DATA_COVERAGE_LOG = build_data_coverage_log(df, train_end=TRAIN_END, test_start=TEST_START)

print('DATA_SOURCE_USED:', DATA_SOURCE_USED)
print('FAIR_VALUE_FEATURES:', FAIR_VALUE_FEATURES)
print('Panel shape:', df.shape)
print('Price outliers flagged:', int(df.get('price_outlier_flag', pd.Series(False, index=df.index)).sum()))
if not banks_universe.empty:
    display_cols = ['bank_id', 'legal_name', 'country', 'status', 'listed_flag', 'ticker', 'group_name', 'bank_category']
    display(banks_universe[[c for c in display_cols if c in banks_universe.columns]].head(20))
display(df.head())
display(DATA_SOURCE_SUMMARY)
display(FUNDAMENTAL_COVERAGE_SUMMARY)
display(DATA_COVERAGE_LOG)
plot_price_cleaning_diagnostics(df, ticker='BMPS.MI')
if PLOTLY_AVAILABLE and PRICE_COL in df.columns:
    fig = px.line(df, x='date', y=PRICE_COL, color='ticker', title='Italian banks clean price panel')
    fig.update_layout(template='plotly_white', height=520)
    fig.show()


## 5. Esperimento 1 - Fair value & mispricing

### Modello peer-implied

Il primo esperimento stima un fair value cross-sectional usando caratteristiche fondamentali e di mercato dei peer bancari.

\[
Y_{i,t} = \log\left( MarketCap_{i,t} \right)
\]

\[
\hat{Y}_{i,t} = f_t(X_{i,t})
\]

\[
\hat{V}_{i,t} = \exp(\hat{Y}_{i,t})
\]

Il mispricing relativo è:

\[
MP_{i,t} = \frac{\hat{V}_{i,t} - V^{mkt}_{i,t}}{V^{mkt}_{i,t}}
\]

La standardizzazione cross-sectional per data è:

\[
z_{i,t} = \frac{MP_{i,t} - \mu_t(MP)}{\sigma_t(MP)}
\]

Interpretazione dei modelli: **OLS** è una baseline lineare; **LASSO** aiuta a selezionare ratio più informativi; **Random Forest** e **Gradient Boosting** catturano non-linearità e interazioni, ma richiedono più attenzione a overfitting e stabilità out-of-sample.


### Model zoo fair value

La versione estesa sotto confronta modelli lineari e non lineari sullo stesso target. I modelli ML non lineari possono catturare interazioni tra ROE, NPL, CET1, BTP/Assets, EVA e multipli di mercato: per esempio una banca con ROE alto ma NPL elevati può essere prezzata diversamente da una banca con lo stesso ROE ma migliore qualità del credito. Le metriche `R2_test` e `IC` vanno lette insieme: il primo misura fit del fair value, il secondo misura se il mispricing ordinato anticipa i rendimenti futuri.


### Diagnostica copertura fondamentali bancari

Il framework prevede feature bancarie come ROE, NPL, CET1, LDR, BTP/Assets ed EVA, ma il panel corrente può non contenerle o contenerle solo per pochi titoli/date. Prima di stimare il fair value, il lab misura la copertura effettiva delle colonne e usa nei modelli solo i fondamentali realmente popolabili. Le feature di mercato restano disponibili come fallback, mentre i fondamentali regolamentari incompleti vengono segnalati invece di essere inventati.


In [ ]:
def inspect_feature_coverage(df: pd.DataFrame, fundamental_cols: list, market_cols: list, min_non_null_ratio: float = 0.2) -> pd.DataFrame:
    """
    Restituisce una tabella con la copertura (% non-null) per ciascuna feature
    e un flag 'usable' per quelle sopra la soglia.
    """
    rows = []
    n = len(df)
    for feature_type, cols in [('fundamental', fundamental_cols), ('market', market_cols)]:
        for col in cols:
            if col in df.columns:
                non_null = int(df[col].notna().sum())
                ratio = float(non_null / n) if n else 0.0
                unique = int(df[col].nunique(dropna=True))
            else:
                non_null, ratio, unique = 0, 0.0, 0
            if ratio == 0:
                status = 'missing_or_empty'
            elif ratio < min_non_null_ratio:
                status = 'sparse'
            else:
                status = 'usable'
            rows.append({
                'feature': col,
                'type': feature_type,
                'exists': col in df.columns,
                'non_null_obs': non_null,
                'non_null_ratio': ratio,
                'unique_values': unique,
                'usable': bool((feature_type == 'market' and col in df.columns and non_null > 0) or (feature_type == 'fundamental' and ratio >= min_non_null_ratio)),
                'status': status,
            })
    out = pd.DataFrame(rows).sort_values(['usable', 'non_null_ratio', 'feature'], ascending=[False, False, True]).reset_index(drop=True)
    return out


def select_usable_features(df: pd.DataFrame, fundamental_cols: list, market_cols: list, min_non_null_ratio: float = 0.2) -> list:
    """
    Ritorna la lista di feature effettivamente usabili nel fair value:
      - includi sempre le market features presenti e non completamente vuote,
      - includi solo le fundamental features con copertura >= soglia.
    """
    coverage = inspect_feature_coverage(df, fundamental_cols, market_cols, min_non_null_ratio)
    usable_fundamentals = coverage.loc[(coverage['type'] == 'fundamental') & coverage['usable'], 'feature'].tolist()
    usable_market = [col for col in market_cols if col in df.columns and df[col].notna().sum() > 0]
    selected = []
    for col in usable_fundamentals + usable_market:
        if col not in selected:
            selected.append(col)
    return selected


FEATURE_COVERAGE_TABLE = inspect_feature_coverage(
    df,
    BANK_FUNDAMENTAL_FEATURES,
    MARKET_FEATURES,
    min_non_null_ratio=float(USER_CONFIG.get('min_feature_non_null_ratio', 0.2)),
)
display(HTML('<h3>Feature coverage diagnostic</h3>'))
display(FEATURE_COVERAGE_TABLE)

FAIR_VALUE_FEATURES_STATIC = list(FAIR_VALUE_FEATURES)
FAIR_VALUE_FEATURES = select_usable_features(
    df,
    BANK_FUNDAMENTAL_FEATURES,
    MARKET_FEATURES,
    min_non_null_ratio=float(USER_CONFIG.get('min_feature_non_null_ratio', 0.2)),
)
EXPECTED_RETURN_FEATURES = [col for col in (FAIR_VALUE_FEATURES + ['mispricing_z', 'mispricing_ens']) if col in df.columns or col in ['mispricing_z', 'mispricing_ens']]
print('Fair value features statiche:', FAIR_VALUE_FEATURES_STATIC)
print('Fair value features usabili:', FAIR_VALUE_FEATURES)
if not FAIR_VALUE_FEATURES:
    raise ValueError('Nessuna feature usabile per il fair value: controlla datacenter, CSV o fallback market data.')


In [ ]:
def _identity_line_bounds(frame, x_col, y_col):
    vals = pd.concat([pd.to_numeric(frame[x_col], errors='coerce'), pd.to_numeric(frame[y_col], errors='coerce')]).dropna()
    vals = vals[vals > 0]
    return (1, 10) if vals.empty else (float(vals.min()), float(vals.max()))


def plot_experiment_diagnostics(fair_value_df=None, ensemble_df=None, quantile_returns=None, long_short_returns=None, title_prefix='Mispricing'):
    """Render fair-value, mispricing and cumulative-return diagnostics with Plotly when available."""
    if not PLOTLY_AVAILABLE:
        if quantile_returns is not None and not quantile_returns.empty:
            plot_cumulative_returns(quantile_returns, title=f'{title_prefix} quantile portfolios')
        if long_short_returns is not None and len(long_short_returns) > 0:
            plot_cumulative_returns(long_short_returns, title=f'{title_prefix} long-short')
        return
    if fair_value_df is not None and not fair_value_df.empty and {MARKET_CAP_COL, 'fair_value'}.issubset(fair_value_df.columns):
        latest = fair_value_df.sort_values('date').groupby('ticker').tail(1).copy()
        latest = latest[(pd.to_numeric(latest[MARKET_CAP_COL], errors='coerce') > 0) & (pd.to_numeric(latest['fair_value'], errors='coerce') > 0)]
        if not latest.empty:
            lo, hi = _identity_line_bounds(latest, MARKET_CAP_COL, 'fair_value')
            fig = px.scatter(latest, x=MARKET_CAP_COL, y='fair_value', color='ticker', hover_name='ticker',
                             hover_data=[c for c in ['company_name', 'mispricing_rel', 'mispricing_z'] if c in latest.columns],
                             log_x=True, log_y=True, title=f'Fair value vs market value · {title_prefix}',
                             labels={MARKET_CAP_COL: 'Clean market value / cap', 'fair_value': 'Peer-implied fair value'})
            fig.add_shape(type='line', x0=lo, y0=lo, x1=hi, y1=hi, line=dict(color='rgba(30,30,30,.55)', dash='dash'))
            fig.update_layout(template='plotly_white', legend_title_text='Ticker', height=560)
            fig.show()
    if ensemble_df is not None and not ensemble_df.empty and 'mispricing_ens' in ensemble_df.columns:
        test = ensemble_df[pd.to_datetime(ensemble_df['date'], errors='coerce') >= pd.Timestamp(TEST_START)].copy()
        if not test.empty:
            fig = px.line(test, x='date', y='mispricing_ens', color='ticker',
                          hover_data=[c for c in ['company_name', 'quantile'] if c in test.columns],
                          title='Mispricing ensemble nel tempo · test set', labels={'mispricing_ens': 'Ensemble z-score'})
            fig.add_hline(y=2, line_dash='dash', line_color='#da7101', annotation_text='+2 z')
            fig.add_hline(y=-2, line_dash='dash', line_color='#da7101', annotation_text='-2 z')
            fig.update_layout(template='plotly_white', height=520, legend_title_text='Ticker')
            fig.show()
    if quantile_returns is not None and not quantile_returns.empty:
        cum = (1 + quantile_returns.fillna(0)).cumprod() - 1
        cum = cum.reset_index().rename(columns={'index': 'date'})
        date_col = 'date' if 'date' in cum.columns else cum.columns[0]
        long = cum.melt(id_vars=date_col, var_name='portfolio', value_name='cumulative_return')
        if long_short_returns is not None and len(long_short_returns) > 0:
            ls = ((1 + long_short_returns.fillna(0)).cumprod() - 1).reset_index()
            ls.columns = [date_col, 'cumulative_return']
            ls['portfolio'] = 'Long-Short'
            long = pd.concat([long, ls[[date_col, 'portfolio', 'cumulative_return']]], ignore_index=True)
        fig = px.line(long, x=date_col, y='cumulative_return', color='portfolio',
                      title=f'Cumulative returns – {title_prefix} quintile portfolios')
        fig.update_yaxes(tickformat='.0%')
        fig.update_traces(hovertemplate='%{x}<br>%{legendgroup}: %{y:.1%}<extra></extra>')
        fig.update_layout(template='plotly_white', height=560, legend_title_text='Portfolio')
        fig.show()


def _make_fair_value_estimator(model_name: str, params: dict):
    model_name = model_name.lower()
    try:
        if model_name == 'ols':
            from sklearn.linear_model import LinearRegression
            return LinearRegression()
        if model_name in {'lasso', 'elastic_net'}:
            from sklearn.linear_model import ElasticNet
            return ElasticNet(random_state=42, max_iter=10000, **{k: v for k, v in params.items() if k != 'enabled'})
        if model_name == 'rf':
            from sklearn.ensemble import RandomForestRegressor
            return RandomForestRegressor(random_state=42, n_jobs=-1, **{k: v for k, v in params.items() if k != 'enabled'})
        if model_name == 'gbrt':
            from sklearn.ensemble import GradientBoostingRegressor
            return GradientBoostingRegressor(random_state=42, **{k: v for k, v in params.items() if k != 'enabled'})
        if model_name == 'xgb':
            try:
                from xgboost import XGBRegressor
                return XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=1, **{k: v for k, v in params.items() if k != 'enabled'})
            except Exception as exc:
                print(f'XGBoost fair-value non disponibile: {exc}')
                return None
        if model_name == 'mlp':
            from sklearn.neural_network import MLPRegressor
            return MLPRegressor(random_state=42, early_stopping=True, **{k: v for k, v in params.items() if k != 'enabled'})
    except Exception as exc:
        print(f'Modello fair-value {model_name} non disponibile: {exc}')
    return None


def _prepare_model_matrix(frame: pd.DataFrame, feature_cols: list):
    X = frame.reindex(columns=feature_cols).astype(float).replace([np.inf, -np.inf], np.nan)
    return X.fillna(X.median(numeric_only=True)).fillna(0.0)


def _regression_metrics(y_true, y_pred):
    y = pd.to_numeric(y_true, errors='coerce')
    p = pd.to_numeric(y_pred, errors='coerce')
    mask = y.notna() & p.notna()
    if mask.sum() == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'mape': np.nan, 'r2': np.nan}
    err = y[mask] - p[mask]
    denom = ((y[mask] - y[mask].mean()) ** 2).sum()
    mape = (err.abs() / y[mask].abs().replace(0, np.nan)).mean()
    return {'rmse': float(np.sqrt((err ** 2).mean())), 'mae': float(err.abs().mean()), 'mape': float(mape) if pd.notna(mape) else np.nan, 'r2': float(1 - (err ** 2).sum() / denom) if denom else np.nan}


def _mean_ic(frame, signal_col, target_col):
    vals = []
    for _, g in frame.groupby('date'):
        if g[signal_col].notna().sum() >= 3 and g[target_col].notna().sum() >= 3:
            vals.append(g[[signal_col, target_col]].corr(method='spearman').iloc[0, 1])
    s = pd.Series(vals).dropna()
    return (float(s.mean()), float(s.std(ddof=0))) if not s.empty else (np.nan, np.nan)


def run_fair_value_models(df: pd.DataFrame, feature_cols: list, target_col: str, model_specs: dict, train_end: pd.Timestamp) -> dict:
    """Addestra più modelli peer-implied e produce predizioni, mispricing e metriche."""
    usable_features = [c for c in feature_cols if c in df.columns and df[c].notna().sum() >= 5]
    train_mask = pd.to_datetime(df['date'], errors='coerce') <= pd.Timestamp(train_end)
    test_mask = pd.to_datetime(df['date'], errors='coerce') > pd.Timestamp(train_end)
    y = pd.to_numeric(df[target_col], errors='coerce')
    X = _prepare_model_matrix(df, usable_features)
    results = {}
    for model_name, params in model_specs.items():
        if not params.get('enabled', True):
            continue
        estimator = _make_fair_value_estimator(model_name, params)
        if estimator is None:
            continue
        valid_train = train_mask & y.notna()
        try:
            estimator.fit(X.loc[valid_train], y.loc[valid_train])
            pred_log = pd.Series(estimator.predict(X), index=df.index, name='pred_log_mcap')
            out = df.copy()
            out['pred_log_mcap'] = pred_log
            out['fair_value'] = np.exp(pred_log)
            out['mispricing_rel'] = (out['fair_value'] - pd.to_numeric(out[MARKET_CAP_COL], errors='coerce')) / pd.to_numeric(out[MARKET_CAP_COL], errors='coerce').replace(0, np.nan)
            out['mispricing_z'] = out.groupby('date')['mispricing_rel'].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) else np.nan)
            out['model_name'] = model_name
            out['used_features'] = ', '.join(usable_features)
            train_m = _regression_metrics(y.loc[train_mask], pred_log.loc[train_mask])
            test_m = _regression_metrics(y.loc[test_mask], pred_log.loc[test_mask])
            ic_mean, ic_std = _mean_ic(out.loc[test_mask].copy(), 'mispricing_z', FORWARD_RETURN_COL)
            metrics_row = {
                'model': model_name,
                'RMSE_train': train_m['rmse'], 'RMSE_test': test_m['rmse'],
                'MAE_train': train_m['mae'], 'MAE_test': test_m['mae'],
                'MAPE_train': train_m['mape'], 'MAPE_test': test_m['mape'],
                'R2_train': train_m['r2'], 'R2_test': test_m['r2'],
                'IC_mean': ic_mean, 'IC_std': ic_std,
                'n_features': len(usable_features),
            }
            results[model_name] = {'estimator': estimator, 'predictions': out, 'metrics': metrics_row, 'features': usable_features}
            print(f"OK fair-value {model_name}: R2_test={metrics_row['R2_test']:.4f}, IC={metrics_row['IC_mean']:.4f}")
        except Exception as exc:
            print(f'SKIP fair-value {model_name}: {exc}')
    return results


fair_value_model_specs = {
    'ols': {'enabled': True},
    'lasso': {'enabled': 'LASSO' in USER_CONFIG.get('fair_value_models', []), 'alpha': 0.001, 'l1_ratio': 0.8},
    'rf': {'enabled': 'Random Forest' in USER_CONFIG.get('fair_value_models', []), **RF_PARAMS},
    'gbrt': {'enabled': 'Gradient Boosting' in USER_CONFIG.get('fair_value_models', []), **GBRT_PARAMS},
    'xgb': {'enabled': True, 'n_estimators': 140, 'max_depth': 3, 'learning_rate': 0.05},
    'mlp': {'enabled': True, 'hidden_layer_sizes': (32, 16), 'alpha': 0.001, 'max_iter': 800},
}

fair_value_model_results = run_fair_value_models(
    df=df,
    feature_cols=FAIR_VALUE_FEATURES,
    target_col=LOG_MCAP_COL,
    model_specs=fair_value_model_specs,
    train_end=pd.Timestamp(TRAIN_END),
)
if not fair_value_model_results:
    raise RuntimeError('Nessun modello fair value è riuscito a girare. Controlla feature e split temporale.')

fair_value_model_metrics = pd.DataFrame([v['metrics'] for v in fair_value_model_results.values()]).sort_values(['R2_test', 'IC_mean'], ascending=False)
legacy_name_map = {'OLS': 'ols', 'LASSO': 'lasso', 'Random Forest': 'rf', 'Gradient Boosting': 'gbrt'}
primary_key = legacy_name_map.get(USER_CONFIG.get('primary_model'), None)
if primary_key not in fair_value_model_results:
    primary_key = fair_value_model_metrics.iloc[0]['model']

fair_value_results = {k: v['predictions'] for k, v in fair_value_model_results.items()}
fair_value_models = {k: v['estimator'] for k, v in fair_value_model_results.items()}
df_rf = fair_value_results[primary_key].copy()
df_gbrt = fair_value_results.get('gbrt', df_rf).copy()
df_ols = fair_value_results.get('ols', df_rf).copy()
df_lasso = fair_value_results.get('lasso', df_rf).copy()
model_rf = fair_value_models[primary_key]
model_gbrt = fair_value_models.get('gbrt', model_rf)
model_ols = fair_value_models.get('ols', model_rf)
model_lasso = fair_value_models.get('lasso', model_rf)

summary_cols = ['date', 'ticker', MARKET_CAP_COL, 'fair_value', 'mispricing_rel', 'mispricing_z', 'model_name']
display(fair_value_model_metrics)
display(df_rf[[col for col in summary_cols if col in df_rf.columns]].tail())
plot_experiment_diagnostics(fair_value_df=df_rf, title_prefix=primary_key)


In [ ]:
def evaluate_fair_value_models(results_dict: dict, return_col: str = "target_ret_1m_fwd") -> pd.DataFrame:
    """
    Prende in input un dict {model_name: df_results} dove df_results contiene:
      - mkt_market_cap_clean, fair_value, mispricing_rel, mispricing_z, target_ret_1m_fwd
    e calcola per ogni modello:
      - RMSE, R2 cross-sectional su log mcap
      - IC tra mispricing_z e target_ret_1m_fwd.
    """
    rows = []
    mcap_candidates = ['mkt_market_cap_clean', MARKET_CAP_COL, 'mkt_market_cap']
    for model_name, model_df in results_dict.items():
        frame = model_df.copy()
        mcap_col = next((c for c in mcap_candidates if c in frame.columns), None)
        if mcap_col is None or 'fair_value' not in frame.columns:
            rows.append({'model': model_name, 'RMSE': np.nan, 'R2_cross_sectional': np.nan, 'IC_mispricing': np.nan, 'n_obs': 0})
            continue
        y = np.log(pd.to_numeric(frame[mcap_col], errors='coerce').where(lambda s: s > 0))
        pred = np.log(pd.to_numeric(frame['fair_value'], errors='coerce').where(lambda s: s > 0))
        mask = y.notna() & pred.notna()
        err = y[mask] - pred[mask]
        denom = ((y[mask] - y[mask].mean()) ** 2).sum()
        rmse = float(np.sqrt((err ** 2).mean())) if mask.sum() else np.nan
        r2 = float(1 - (err ** 2).sum() / denom) if mask.sum() and denom else np.nan
        if {'mispricing_z', return_col}.issubset(frame.columns):
            ic_values = []
            for _, g in frame.groupby('date'):
                valid = g[['mispricing_z', return_col]].replace([np.inf, -np.inf], np.nan).dropna()
                if len(valid) >= 3:
                    ic_values.append(valid.corr(method='spearman').iloc[0, 1])
            ic = float(pd.Series(ic_values).dropna().mean()) if ic_values else np.nan
        else:
            ic = np.nan
        rows.append({'model': model_name, 'RMSE': rmse, 'R2_cross_sectional': r2, 'IC_mispricing': ic, 'n_obs': int(mask.sum())})
    return pd.DataFrame(rows).sort_values(['R2_cross_sectional', 'IC_mispricing'], ascending=False).reset_index(drop=True)


fair_value_comparison_metrics = evaluate_fair_value_models(fair_value_results, return_col=FORWARD_RETURN_COL)
display(HTML('<h3>Fair value model comparison · compact</h3>'))
display(fair_value_comparison_metrics)


## 6. Esperimento 2 - Mispricing ensemble & quintile portfolios

L'ensemble combina i segnali standardizzati dei modelli disponibili:

\[
MP^{ens}_{i,t} = \frac{1}{K}\sum_k z^{(k)}_{i,t}
\]

A ogni data il segnale viene ordinato in quantili: i titoli nel quantile alto sono quelli più sottovalutati secondo il modello, quelli nel quantile basso i più cari. La strategia long-short compra il quantile alto e vende il quantile basso, ad esempio Q5-Q1 quando ci sono cinque quantili.

Questo è un test di qualità del segnale: se il mispricing è informativo, i portafogli ordinati dal segnale dovrebbero mostrare monotonicità e il long-short dovrebbe avere rendimento medio e Sharpe positivi. In seguito, l'esperimento expected-return usa invece un ordinamento **prediction-sorted**, cioè basato direttamente sulla previsione del rendimento futuro.


In [ ]:
df_ens_base = df_rf.copy()
ensemble_inputs = []
for model_name, model_df in fair_value_results.items():
    col = 'mispricing_z_' + str(model_name).lower().replace(' ', '_')
    df_ens_base[col] = model_df['mispricing_z'].reindex(df_ens_base.index)
    ensemble_inputs.append((col, df_ens_base[col]))

if ENSEMBLE_ENABLED and len(ensemble_inputs) > 1:
    ensemble = EnsembleMispricingSignal(
        input_signals=ensemble_inputs,
        combine_method='zmean',
        out_col='mispricing_ens',
    )
    df_ens = ensemble.combine(base_df=df_ens_base)
else:
    df_ens = df_ens_base.copy()
    source_col = ensemble_inputs[0][0] if ensemble_inputs else 'mispricing_z'
    df_ens['mispricing_ens'] = df_ens[source_col]
df_ens = QuantileSorter(signal_col='mispricing_ens', n_quantiles=QUANTILES).assign_quantiles(df_ens)
df_ens_test = df_ens[df_ens['date'] >= pd.Timestamp(TEST_START)].copy()

builder = QuantilePortfolioBuilder(
    return_col=FORWARD_RETURN_COL,
    quantile_col='quantile',
    weighting=USER_CONFIG.get('portfolio_weighting', 'equal'),
)

quintile_returns = builder.build(df_ens_test)
long_short_returns = builder.long_short(
    quintile_returns,
    long_q=QUANTILES,
    short_q=1,
).rename('mispricing_ls')

print(quintile_returns.tail())
plot_experiment_diagnostics(ensemble_df=df_ens, quantile_returns=quintile_returns, long_short_returns=long_short_returns, title_prefix='mispricing')
if PLOTLY_AVAILABLE:
    latest = df_ens.sort_values('date').groupby('ticker').tail(1)
    fig = px.bar(latest.sort_values('mispricing_ens'), x='ticker', y='mispricing_ens', color='quantile', title='Latest ensemble mispricing by bank', labels={'mispricing_ens': 'Ensemble z-score'})
    fig.update_layout(template='plotly_white', height=460)
    fig.show()


## 7. Esperimento 3 - Screening bancario con filtri regolamentari

Applichiamo soglie CET1/NPL/liquidita', quindi costruiamo uno score finale filtrato e portafogli ordinati per quintile.

In [ ]:
screen = ScreeningFunction(
    cet1_col='fund_cet1_lag',
    npl_col='fund_npl_ratio_lag',
    liq_col='fund_ldr_lag',
    cet1_min=SCREENING_PARAMS['cet1_min'],
    npl_max=SCREENING_PARAMS['npl_max'],
    liq_min=SCREENING_PARAMS['liq_min'],
    out_col='screen_pass',
)

missing_screen_cols = [col for col in ['fund_cet1_lag', 'fund_npl_ratio_lag', 'fund_ldr_lag'] if col not in df_ens.columns or df_ens[col].notna().sum() == 0]
if missing_screen_cols:
    print('Screening regolamentare parziale: colonne mancanti/non disponibili:', missing_screen_cols)
    df_screen = df_ens.copy()
    df_screen['screen_pass'] = True
    df_screen['screening_data_quality'] = 'market_only_no_regulatory_fundamentals'
else:
    df_screen = screen.apply(df_ens)
    df_screen['screening_data_quality'] = 'regulatory_screen_applied'

df_screen['score_fund'] = df_screen['mispricing_ens']
df_screen['score_final'] = df_screen['screen_pass'].astype(int) * df_screen['score_fund']

df_screen = QuantileSorter(signal_col='score_final', n_quantiles=QUANTILES).assign_quantiles(df_screen)
df_screen_test = df_screen[df_screen['date'] >= pd.Timestamp(TEST_START)].copy()

screen_quintile_returns = builder.build(df_screen_test)
screen_ls_returns = builder.long_short(screen_quintile_returns, long_q=QUANTILES, short_q=1).rename('screening_ls')

cols = ['date', 'ticker', 'screen_pass', 'screening_data_quality', 'score_final', 'quantile']
display(df_screen[[col for col in cols if col in df_screen.columns]].tail())
plot_cumulative_returns(screen_ls_returns, title='Screening LS - Italian banks')


## 8. Esperimento 4 - Expected returns ML

Qui il target non è il fair value ma il rendimento futuro mensile:

\[
r_{i,t+1} = \log\left(\frac{P_{i,t+1}}{P_{i,t}}\right)
\]

Il modello predice:

\[
\hat{r}_{i,t+1} = f_t(Z_{i,t})
\]

con feature fondamentali, tecniche e il segnale di mispricing. La metrica out-of-sample è:

\[
R^2_{OS} = 1 - \frac{\sum (r_{i,t+1} - \hat{r}_{i,t+1})^2}{\sum (r_{i,t+1} - \bar{r}_{train})^2}
\]

Un \(R^2_{OS}\) negativo significa che il modello è peggiore della media storica di training: può indicare overfitting, regime shift o segnale insufficiente nel campione disponibile. In finanza questo è frequente, quindi il risultato va letto insieme a robustezza, turnover e performance dei portafogli ordinati.


### Model zoo expected returns

Da qui in avanti il lab confronta più famiglie di modello: baseline lineare regolarizzata, Random Forest, Gradient Boosting, XGBoost se disponibile e MLP. Su un universo piccolo e regolamentato come le banche italiane, questo confronto è utile perché i modelli lineari sono più stabili e interpretabili, mentre i non lineari possono catturare interazioni tra value, momentum, leverage e qualità. La tabella finale va letta congiuntamente: un buon \(R^2_{OS}\) senza performance long-short può non essere investibile, mentre una strategia LS positiva con \(R^2_{OS}\) negativo può essere instabile o guidata da pochi outlier.


Un modello lineare regolarizzato come Elastic Net/LASSO è la baseline robusta: è meno espressivo di RF/GBRT/MLP, ma tende a essere più stabile quando l'universo è piccolo e i dati sono rumorosi. Un \(R^2_{OS}\) moderatamente negativo su 40-60 mesi out-of-sample non squalifica automaticamente il modello: segnala che la previsione puntuale è debole, mentre il contenuto economico può emergere meglio nei portafogli ordinati per previsione e nei test di stabilità.


In [ ]:
usable_pred_features = [col for col in (FAIR_VALUE_FEATURES + ['mispricing_ens']) if col in df_ens.columns and df_ens[col].notna().sum() >= 5]
if not usable_pred_features:
    raise ValueError('Nessuna feature di expected return disponibile nel dataset.')


def _make_expected_return_estimator(model_name: str, params: dict):
    model_name = model_name.lower()
    try:
        if model_name == 'rf':
            from sklearn.ensemble import RandomForestRegressor
            return RandomForestRegressor(random_state=42, n_jobs=-1, **{k: v for k, v in params.items() if k != 'enabled'})
        if model_name == 'gbrt':
            from sklearn.ensemble import GradientBoostingRegressor
            return GradientBoostingRegressor(random_state=42, **{k: v for k, v in params.items() if k != 'enabled'})
        if model_name == 'xgb':
            try:
                from xgboost import XGBRegressor
                return XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=1, **{k: v for k, v in params.items() if k != 'enabled'})
            except Exception as exc:
                print(f'XGBoost non disponibile, skip: {exc}')
                return None
        if model_name == 'mlp':
            from sklearn.neural_network import MLPRegressor
            return MLPRegressor(random_state=42, early_stopping=True, **{k: v for k, v in params.items() if k != 'enabled'})
        if model_name == 'lasso':
            from sklearn.linear_model import ElasticNet
            return ElasticNet(random_state=42, max_iter=10000, **{k: v for k, v in params.items() if k != 'enabled'})
    except Exception as exc:
        print(f'Modello {model_name} non disponibile: {exc}')
    return None


def _prepare_model_matrix(frame: pd.DataFrame, feature_cols: list):
    X = frame.reindex(columns=feature_cols).astype(float).replace([np.inf, -np.inf], np.nan)
    return X.fillna(X.median(numeric_only=True)).fillna(0.0)


def _hit_ratio(y_true, y_pred):
    y = pd.to_numeric(y_true, errors='coerce')
    p = pd.to_numeric(y_pred, errors='coerce')
    mask = y.notna() & p.notna() & y.ne(0) & p.ne(0)
    return float((np.sign(y[mask]) == np.sign(p[mask])).mean()) if mask.sum() else np.nan


def _mean_cross_sectional_ic(frame, pred_col, target_col):
    rows = []
    for date, g in frame.groupby('date'):
        if g[pred_col].notna().sum() >= 3 and g[target_col].notna().sum() >= 3:
            rows.append(g[[pred_col, target_col]].corr(method='spearman').iloc[0, 1])
    return float(pd.Series(rows).dropna().mean()) if rows else np.nan


def run_expected_return_models(df: pd.DataFrame, feature_cols: list, target_col: str, train_end: pd.Timestamp, model_specs: dict) -> dict:
    """Train multiple expected-return models and build prediction-sorted portfolios."""
    train_mask = pd.to_datetime(df['date'], errors='coerce') <= pd.Timestamp(train_end)
    test_mask = pd.to_datetime(df['date'], errors='coerce') > pd.Timestamp(train_end)
    train = df.loc[train_mask].copy()
    test = df.loc[test_mask].copy()
    y_train = pd.to_numeric(train[target_col], errors='coerce')
    valid_train = y_train.notna()
    X_train = _prepare_model_matrix(train.loc[valid_train], feature_cols)
    y_train = y_train.loc[valid_train]
    benchmark = y_train.mean()
    results = {}
    q_builder = QuantilePortfolioBuilder(return_col=target_col, quantile_col='quantile', weighting=USER_CONFIG.get('portfolio_weighting', 'equal'))
    for model_name, params in model_specs.items():
        if not params.get('enabled', True):
            continue
        estimator = _make_expected_return_estimator(model_name, params)
        if estimator is None or X_train.empty:
            continue
        try:
            estimator.fit(X_train, y_train)
            pred = df.copy()
            pred_col = f'pred_ret_1m_fwd_{model_name}'
            pred[pred_col] = estimator.predict(_prepare_model_matrix(pred, feature_cols))
            pred_test = pred.loc[test_mask].copy()
            pred_test = QuantileSorter(signal_col=pred_col, n_quantiles=QUANTILES).assign_quantiles(pred_test)
            q_returns = q_builder.build(pred_test)
            ls = q_builder.long_short(q_returns, long_q=QUANTILES, short_q=1).rename(f'{model_name}_prediction_ls')
            perf = PerformanceMetrics(ls).summary().iloc[0].to_dict() if len(ls) else {}
            metrics_row = {
                'model': model_name,
                'R2_OS': oos_r2(pred_test[target_col], pred_test[pred_col], benchmark=benchmark),
                'hit_ratio': _hit_ratio(pred_test[target_col], pred_test[pred_col]),
                'IC': _mean_cross_sectional_ic(pred_test, pred_col, target_col),
                'mean_ls_return': perf.get('mean_return', np.nan),
                'sharpe_ls': perf.get('sharpe', np.nan),
                'max_drawdown_ls': perf.get('max_drawdown', np.nan),
                'periods': perf.get('periods', 0),
            }
            results[model_name] = {'estimator': estimator, 'predictions': pred, 'predictions_test': pred_test, 'metrics': metrics_row, 'ports': q_returns, 'ls': ls, 'pred_col': pred_col}
            print(f"OK expected-return {model_name}: R2_OS={metrics_row['R2_OS']:.4f}, Sharpe={metrics_row['sharpe_ls']:.2f}")
        except Exception as exc:
            print(f'SKIP expected-return {model_name}: {exc}')
    return results


expected_return_results = run_expected_return_models(
    df=df_ens,
    feature_cols=usable_pred_features,
    target_col=FORWARD_RETURN_COL,
    train_end=pd.Timestamp(TRAIN_END),
    model_specs=USER_CONFIG.get('expected_return_model_specs', {}),
)

if not expected_return_results:
    raise RuntimeError('Nessun modello expected-return è riuscito a girare.')

expected_return_model_metrics = pd.DataFrame([v['metrics'] for v in expected_return_results.values()]).sort_values(['R2_OS', 'sharpe_ls'], ascending=False)
best_prediction_model = expected_return_model_metrics.iloc[0]['model']
best_prediction = expected_return_results[best_prediction_model]
predictions_df = best_prediction['predictions']
predictions_test = best_prediction['predictions_test']
pred_quintile_returns = best_prediction['ports']
pred_ls_returns = best_prediction['ls'].rename('prediction_ls')
r2_os = float(expected_return_model_metrics.iloc[0]['R2_OS'])

print(f'Best expected-return model: {best_prediction_model} · R2_OS={r2_os:.4f}')
display(expected_return_model_metrics)
if PLOTLY_AVAILABLE:
    fig = px.bar(expected_return_model_metrics, x='model', y=['R2_OS', 'hit_ratio', 'IC'], barmode='group', title='Expected-return model comparison')
    fig.update_layout(template='plotly_white', height=460)
    fig.show()
plot_experiment_diagnostics(quantile_returns=pred_quintile_returns, long_short_returns=pred_ls_returns, title_prefix=f'prediction · {best_prediction_model}')


In [ ]:
def compare_expected_return_models(df: pd.DataFrame, feature_cols: list, target_col: str, train_end: pd.Timestamp) -> pd.DataFrame:
    """
    Confronta almeno due modelli:
      - 'lasso' (lineare regolarizzato)
      - 'rf' (Random Forest)
    con metriche:
      - R2_OS
      - hit_ratio
      - LS_mean_return, LS_sharpe
    """
    specs = {
        'lasso': {'enabled': True, 'alpha': 0.001, 'l1_ratio': 0.8},
        'rf': {'enabled': True, **USER_CONFIG.get('expected_return_rf_params', USER_CONFIG.get('rf_params', {}))},
    }
    if 'run_expected_return_models' in globals():
        results = run_expected_return_models(df, feature_cols, target_col, train_end, specs)
        rows = []
        for model_name, payload in results.items():
            row = dict(payload.get('metrics', {}))
            row['model'] = model_name
            row['LS_mean_return'] = row.get('mean_ls_return', np.nan)
            row['LS_sharpe'] = row.get('sharpe_ls', np.nan)
            rows.append(row)
        cols = ['model', 'R2_OS', 'hit_ratio', 'IC', 'LS_mean_return', 'LS_sharpe', 'max_drawdown_ls', 'periods']
        return pd.DataFrame(rows).reindex(columns=cols).sort_values(['R2_OS', 'LS_sharpe'], ascending=False).reset_index(drop=True)
    raise RuntimeError('run_expected_return_models non è ancora disponibile: esegui prima la cella expected returns.')


expected_return_baseline_comparison = compare_expected_return_models(
    df=df_ens,
    feature_cols=usable_pred_features,
    target_col=FORWARD_RETURN_COL,
    train_end=pd.Timestamp(TRAIN_END),
)
display(HTML('<h3>Expected return baseline comparison · LASSO vs RF</h3>'))
display(expected_return_baseline_comparison)


## 9. Strategia ibrida selection + timing tecnico

La tesi non si ferma allo stock picking: combina selezione fondamentale e timing tecnico. La selezione (`Score_sel`) misura quali banche appaiono più interessanti tra i peer, mentre il timing (`Signal_tech`) prova a stimare se il regime tecnico del titolo/settore è favorevole.

\[
Signal_{tot} = \lambda z(Score_{sel}) + (1 - \lambda) z(Signal_{tech})
\]

Il timing può migliorare il profilo rischio/rendimento quando riduce esposizione nei drawdown o nei regimi momentum negativi. Può anche peggiorare i risultati se introduce turnover, rumore e falsi segnali; per questo il confronto con sola selezione fondamentale resta essenziale.


In [ ]:
HYBRID_LAMBDA = float(USER_CONFIG.get('hybrid_lambda', 0.55))


def add_technical_indicators(panel: pd.DataFrame, price_col: str = PRICE_COL) -> pd.DataFrame:
    """Add MACD, RSI, stochastic proxy, moving-average and drawdown features per ticker."""
    out = panel.sort_values(['ticker', 'date']).copy()
    def _per_ticker(g):
        pxs = pd.to_numeric(g[price_col], errors='coerce')
        ema12 = pxs.ewm(span=12, adjust=False, min_periods=4).mean()
        ema26 = pxs.ewm(span=26, adjust=False, min_periods=8).mean()
        g['macd'] = ema12 - ema26
        g['macd_signal'] = g['macd'].ewm(span=9, adjust=False, min_periods=4).mean()
        delta = pxs.diff()
        gain = delta.clip(lower=0).rolling(14, min_periods=5).mean()
        loss = (-delta.clip(upper=0)).rolling(14, min_periods=5).mean()
        rs = gain / loss.replace(0, np.nan)
        g['rsi'] = 100 - (100 / (1 + rs))
        low14 = pxs.rolling(14, min_periods=5).min()
        high14 = pxs.rolling(14, min_periods=5).max()
        g['stoch_k'] = 100 * (pxs - low14) / (high14 - low14).replace(0, np.nan)
        g['ma_50'] = pxs.rolling(10, min_periods=4).mean()   # monthly proxy ~50 trading days
        g['ma_200'] = pxs.rolling(40, min_periods=12).mean() # monthly proxy ~200 trading days
        g['ma_50_200_spread'] = g['ma_50'] / g['ma_200'].replace(0, np.nan) - 1
        roll6 = pxs.rolling(6, min_periods=3).max()
        roll12 = pxs.rolling(12, min_periods=4).max()
        g['drawdown_6m'] = pxs / roll6 - 1
        g['drawdown_12m'] = pxs / roll12 - 1
        return g
    return out.groupby('ticker', group_keys=False).apply(_per_ticker)


def build_technical_timing_signal(df: pd.DataFrame, target_col: str = FORWARD_RETURN_COL, train_end: str = TRAIN_END) -> pd.DataFrame:
    """Estimate a simple technical timing probability using logistic regression where possible."""
    tech_cols = ['macd', 'macd_signal', 'rsi', 'stoch_k', 'ma_50_200_spread', 'drawdown_6m', 'drawdown_12m']
    out = add_technical_indicators(df)
    available = [c for c in tech_cols if c in out.columns and out[c].notna().sum() >= 8]
    out['signal_tech'] = out[available].rank(pct=True).mean(axis=1) if available else 0.0
    if not available or target_col not in out.columns:
        return out
    train_mask = pd.to_datetime(out['date'], errors='coerce') <= pd.Timestamp(train_end)
    y = (pd.to_numeric(out[target_col], errors='coerce') > 0).astype(float)
    X = _prepare_model_matrix(out, available)
    valid = train_mask & y.notna()
    if valid.sum() >= 30 and y.loc[valid].nunique() > 1:
        try:
            from sklearn.linear_model import LogisticRegression
            clf = LogisticRegression(max_iter=1000, C=0.5)
            clf.fit(X.loc[valid], y.loc[valid])
            out['signal_tech'] = clf.predict_proba(X)[:, 1]
            out['signal_tech_model'] = 'logit_technical'
        except Exception as exc:
            print(f'Timing logit fallback rank-score: {exc}')
            out['signal_tech_model'] = 'rank_proxy'
    else:
        out['signal_tech_model'] = 'rank_proxy_insufficient_train'
    return out


def _z_by_date(series: pd.Series, dates: pd.Series) -> pd.Series:
    frame = pd.DataFrame({'value': pd.to_numeric(series, errors='coerce'), 'date': dates})
    return frame.groupby('date')['value'].transform(lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) else 0.0)


def build_hybrid_signal(score_sel: pd.Series, signal_tech: pd.Series, dates: pd.Series | None = None, lam: float = 0.5) -> pd.Series:
    """Combina selezione fondamentale e timing tecnico: lam*z(selection)+(1-lam)*z(timing)."""
    if dates is None:
        z_sel = (score_sel - score_sel.mean()) / score_sel.std(ddof=0)
        z_tech = (signal_tech - signal_tech.mean()) / signal_tech.std(ddof=0)
    else:
        z_sel = _z_by_date(score_sel, dates)
        z_tech = _z_by_date(signal_tech, dates)
    return (lam * z_sel.fillna(0.0) + (1 - lam) * z_tech.fillna(0.0)).rename('signal_tot')


hybrid_base = build_technical_timing_signal(df_screen.copy(), target_col=FORWARD_RETURN_COL, train_end=TRAIN_END)
hybrid_base['score_sel'] = pd.to_numeric(hybrid_base.get('score_final', hybrid_base.get('mispricing_ens')), errors='coerce')
hybrid_base['signal_tot'] = build_hybrid_signal(hybrid_base['score_sel'], hybrid_base['signal_tech'], hybrid_base['date'], lam=HYBRID_LAMBDA)
hybrid_base = QuantileSorter(signal_col='signal_tot', n_quantiles=QUANTILES).assign_quantiles(hybrid_base)
hybrid_test = hybrid_base[pd.to_datetime(hybrid_base['date'], errors='coerce') >= pd.Timestamp(TEST_START)].copy()

hybrid_builder = QuantilePortfolioBuilder(return_col=FORWARD_RETURN_COL, quantile_col='quantile', weighting=USER_CONFIG.get('portfolio_weighting', 'equal'))
hybrid_quintile_returns = hybrid_builder.build(hybrid_test)
hybrid_ls_returns = hybrid_builder.long_short(hybrid_quintile_returns, long_q=QUANTILES, short_q=1).rename('hybrid_selection_timing_ls')
technical_only = QuantileSorter(signal_col='signal_tech', n_quantiles=QUANTILES).assign_quantiles(hybrid_base)
technical_test = technical_only[pd.to_datetime(technical_only['date'], errors='coerce') >= pd.Timestamp(TEST_START)].copy()
technical_quintile_returns = hybrid_builder.build(technical_test)
technical_ls_returns = hybrid_builder.long_short(technical_quintile_returns, long_q=QUANTILES, short_q=1).rename('technical_timing_ls')

hybrid_metrics = pd.concat([
    PerformanceMetrics(screen_ls_returns).summary().assign(strategy='selection_only'),
    PerformanceMetrics(technical_ls_returns).summary().assign(strategy='technical_only'),
    PerformanceMetrics(hybrid_ls_returns).summary().assign(strategy='hybrid_selection_timing'),
], ignore_index=True)
display(hybrid_metrics)
plot_experiment_diagnostics(quantile_returns=hybrid_quintile_returns, long_short_returns=hybrid_ls_returns, title_prefix='hybrid selection + timing')


## 9. Valutazione & report finale

Sintesi delle metriche principali e, se disponibile, regressione alpha contro fattori esterni.

In [ ]:
def summarize_strategy_metrics(metrics_df: pd.DataFrame, min_sharpe_obs: int = 24) -> pd.DataFrame:
    """Rende leggibili le metriche di strategia e aggiunge commenti interpretativi."""
    out = metrics_df.copy()
    if 'periods' in out.columns:
        out['n_obs'] = pd.to_numeric(out['periods'], errors='coerce').fillna(0).astype(int)
    else:
        out['n_obs'] = np.nan
    out['ann_return_pct'] = 100 * pd.to_numeric(out.get('annualized_return', np.nan), errors='coerce')
    if 'alpha_ann' in out.columns:
        out['ann_alpha_pct'] = 100 * pd.to_numeric(out['alpha_ann'], errors='coerce')
    else:
        out['ann_alpha_pct'] = np.nan
    out['sharpe_readable'] = pd.to_numeric(out.get('sharpe', np.nan), errors='coerce')
    out.loc[out['n_obs'] < min_sharpe_obs, 'sharpe_readable'] = np.nan
    comments = []
    for _, row in out.iterrows():
        note = []
        if pd.notna(row.get('n_obs')) and row['n_obs'] < 60:
            note.append(f"campione oos breve ({int(row['n_obs'])} mesi), interpretare con cautela")
        if pd.isna(row.get('sharpe_readable')):
            note.append('Sharpe non robusto/non calcolabile')
        if pd.notna(row.get('ann_alpha_pct')) and abs(row['ann_alpha_pct']) < 5:
            note.append('alpha economicamente piccolo; verificare IC/intervalli')
        comments.append('; '.join(note) if note else 'lettura standard')
    out['comment'] = comments
    display_cols = [c for c in ['strategy', 'n_obs', 'ann_return_pct', 'ann_alpha_pct', 'annualized_vol', 'sharpe_readable', 'max_drawdown', 'comment'] if c in out.columns]
    return out[display_cols]


def subperiod_metrics(return_series_map: dict[str, pd.Series]) -> pd.DataFrame:
    """Split OOS in two halves and recompute basic metrics."""
    rows = []
    for name, series in return_series_map.items():
        r = pd.to_numeric(series, errors='coerce').dropna()
        if r.empty:
            continue
        midpoint = len(r) // 2
        chunks = {'first_half': r.iloc[:midpoint], 'second_half': r.iloc[midpoint:]}
        for period, chunk in chunks.items():
            if chunk.empty:
                continue
            m = PerformanceMetrics(chunk).summary().iloc[0].to_dict()
            rows.append({'strategy': name, 'subperiod': period, **m})
    return pd.DataFrame(rows)


def bootstrap_ls_mean(series: pd.Series, n_boot: int = 1000, seed: int = 42) -> dict:
    """Empirical bootstrap confidence interval for mean LS return."""
    r = pd.to_numeric(series, errors='coerce').dropna().to_numpy()
    if r.size == 0:
        return {'mean': np.nan, 'ci_5': np.nan, 'ci_95': np.nan, 'n_obs': 0}
    rng = np.random.default_rng(seed)
    samples = rng.choice(r, size=(n_boot, r.size), replace=True).mean(axis=1)
    return {'mean': float(r.mean()), 'ci_5': float(np.percentile(samples, 5)), 'ci_95': float(np.percentile(samples, 95)), 'n_obs': int(r.size)}


metrics_mp = PerformanceMetrics(long_short_returns).summary().assign(strategy='mispricing_ls')
metrics_screen = PerformanceMetrics(screen_ls_returns).summary().assign(strategy='screening_ls')
metrics_pred = PerformanceMetrics(pred_ls_returns).summary().assign(strategy=f'prediction_ls_{best_prediction_model}')
metrics_hybrid = PerformanceMetrics(hybrid_ls_returns).summary().assign(strategy='hybrid_selection_timing_ls')
metrics_tech = PerformanceMetrics(technical_ls_returns).summary().assign(strategy='technical_timing_ls')

experiment_config_table = summarize_experiment_config(USER_CONFIG)
display(HTML('<h3>Experiment config</h3>'))
display(experiment_config_table)

display(HTML('<h3>Fair-value model metrics</h3>'))
display(fair_value_model_metrics)

display(HTML('<h3>Expected-return model metrics</h3>'))
display(expected_return_model_metrics)

metrics = pd.concat([metrics_mp, metrics_screen, metrics_pred, metrics_hybrid, metrics_tech], ignore_index=True)
metrics = metrics[['strategy', 'periods', 'mean_return', 'annualized_return', 'annualized_vol', 'sharpe', 'max_drawdown']]
display(HTML('<h3>Strategy metrics · raw</h3>'))
display(metrics)
display(HTML('<h3>Strategy metrics · interpreted</h3>'))
strategy_metrics_interpreted = summarize_strategy_metrics(metrics)
display(strategy_metrics_interpreted)

return_series_map = {
    'mispricing_ls': long_short_returns,
    'screening_ls': screen_ls_returns,
    f'prediction_ls_{best_prediction_model}': pred_ls_returns,
    'hybrid_selection_timing_ls': hybrid_ls_returns,
    'technical_timing_ls': technical_ls_returns,
}
metrics_subperiods = subperiod_metrics(return_series_map)
bootstrap_summary = pd.DataFrame([{'strategy': k, **bootstrap_ls_mean(v)} for k, v in return_series_map.items()])
display(HTML('<h3>Subperiod stability</h3>'))
display(metrics_subperiods)
display(HTML('<h3>Bootstrap LS mean return · empirical CI 5%-95%</h3>'))
display(bootstrap_summary)

display(HTML('<h3>Data source / fundamentals coverage</h3>'))
display(DATA_SOURCE_SUMMARY)
display(FUNDAMENTAL_COVERAGE_SUMMARY)
display(DATA_COVERAGE_LOG)

if PLOTLY_AVAILABLE:
    compare = pd.concat([
        long_short_returns.rename('mispricing_ls'),
        screen_ls_returns.rename('screening_ls'),
        pred_ls_returns.rename(f'prediction_ls_{best_prediction_model}'),
        hybrid_ls_returns.rename('hybrid_selection_timing_ls'),
        technical_ls_returns.rename('technical_timing_ls'),
    ], axis=1).fillna(0)
    cum = (1 + compare).cumprod() - 1
    cum = cum.reset_index()
    x_col = 'date' if 'date' in cum.columns else cum.columns[0]
    plot = cum.melt(id_vars=x_col, var_name='strategy', value_name='cumulative_return')
    fig = px.line(plot, x=x_col, y='cumulative_return', color='strategy', title='Final LS comparison · thesis strategies')
    fig.update_yaxes(tickformat='.0%')
    fig.update_layout(template='plotly_white', height=620)
    fig.show()

plot_experiment_diagnostics(
    fair_value_df=df_rf,
    ensemble_df=df_ens,
    quantile_returns=quintile_returns,
    long_short_returns=long_short_returns,
    title_prefix='final report',
)


## Interpretazione delle metriche

Un alpha annualizzato di -2.4% con circa 52 osservazioni out-of-sample non va letto come una verità puntuale: con un campione così breve e un universo concentrato può non essere statisticamente diverso da zero. In questa tesi la chiave è la **stabilità**: sottoperiodi, bootstrap e monotonicità dei portafogli sorted contano più del singolo numero di Sharpe o alpha. Anche \(R^2_{OS}\) e Sharpe sono molto sensibili a pochi outlier, specialmente con banche italiane e reverse split storici. Per questo il notebook affianca metriche raw, commenti interpretativi, subperiod test e intervalli bootstrap.

Limiti principali: piccola dimensione del campione, pochi cicli oos, universo concentrato, fondamentali regolamentari incompleti e proxy alternativi ancora imperfetti. Estensioni naturali: import di CSV regolamentari completi, peer europei armonizzati, fattori AQR/Fama-French, costi di transazione e modelli di regime settoriale più robusti.


## 10. Discussione dei risultati

### Lettura in stile paper

La tesi empirica va letta attraverso cinque ipotesi operative:

- **H1 - ML fair value vs OLS:** se RF/GBRT/XGB/MLP migliorano `R2_test` o `IC_mean` rispetto a OLS, il pricing bancario contiene interazioni non lineari tra capitale, qualità credito, redditività e multipli. Se non migliorano, OLS resta una baseline più robusta e interpretabile.
- **H2 - Potere dei ratio bancari:** feature come ROE, NPL, CET1, LDR, BTP/Assets ed EVA dovrebbero rendere il mispricing più stabile rispetto a soli multipli di mercato. Quando mancano dati regolamentari, i proxy `_alt` servono come bridge, ma non sostituiscono la disclosure ufficiale.
- **H3 - Screener regolamentare:** il filtro CET1/NPL/LDR dovrebbe ridurre nomi fragili e migliorare drawdown/Sharpe. Se non accade, può indicare che il campione è troppo piccolo o che i proxy regolamentari non sono abbastanza informativi.
- **H4 - ML expected returns:** `R2_OS` può essere modesto o negativo su un universo piccolo; perciò la prova principale è la qualità dei portafogli prediction-sorted e non la previsione puntuale.
- **H5 - Selection + timing:** il segnale ibrido dovrebbe migliorare il profilo rischio/rendimento quando il timing tecnico evita regimi sfavorevoli. Va però controllato per turnover e costi di transazione.

### Limiti

Il laboratorio resta sensibile a copertura dati, reverse split/prezzi anomali, dimensione ridotta dell'universo, pochi cicli out-of-sample e qualità dei fondamentali alternativi. Le estensioni naturali sono: import CSV regolamentari completi, peer europei armonizzati, costi di transazione, bootstrap/subperiod tests, fattori AQR/Fama-French e un regime model settoriale più robusto.


In [ ]:
# Se hai un factor_df date-indexed gia' disponibile, decommenta e sostituisci i fattori.
# factor_df = pd.read_csv('/content/factors.csv', parse_dates=['date']).set_index('date')

if 'factor_df' in globals():
    alpha_eval = FactorAlphaEvaluator(
        portfolio_returns=long_short_returns,
        factor_data=factor_df,
    )
    alpha_results = alpha_eval.run()
else:
    alpha_eval = FactorAlphaEvaluator(portfolio_returns=long_short_returns)
    alpha_results = alpha_eval.run()

alpha_results

## 10. Parametrizzazione e riuso

Wrapper per eseguire l'intera pipeline cambiando modello fair value, modello expected returns, feature e soglie.

In [ ]:
def run_full_experiment(
    df: pd.DataFrame,
    fair_value_model_class=PeerImpliedRF,
    expected_return_model_class=FundamentalPredictorRF,
    fair_value_features: list[str] = FAIR_VALUE_FEATURES,
    expected_return_features: list[str] = EXPECTED_RETURN_FEATURES,
    train_end: str = TRAIN_END,
    test_start: str = TEST_START,
    screening_params: dict = SCREENING_PARAMS,
) -> dict:
    '''Esegue fair value, mispricing, screening, expected returns e metriche.'''
    fv_df, fv_model = run_fair_value_experiment(
        df=df,
        model_class=fair_value_model_class,
        feature_cols=fair_value_features,
        target_col=LOG_MCAP_COL,
        train_end=train_end,
    )

    fv_df = fv_df.rename(columns={'mispricing_z': 'mispricing_model_z'})
    ens = EnsembleMispricingSignal(
        input_signals=[('mispricing_model_z', fv_df['mispricing_model_z'])],
        combine_method='zmean',
        out_col='mispricing_ens',
    ).combine(base_df=fv_df)

    ens = QuantileSorter('mispricing_ens', 5).assign_quantiles(ens)
    ens_test = ens[ens['date'] >= pd.Timestamp(test_start)].copy()
    q_builder = QuantilePortfolioBuilder(return_col=FORWARD_RETURN_COL, quantile_col='quantile')
    q_returns = q_builder.build(ens_test)
    mp_ls = q_builder.long_short(q_returns, 5, 1).rename('mispricing_ls')

    scr = ScreeningFunction(**screening_params).apply(ens)
    scr['score_fund'] = scr['mispricing_ens']
    scr['score_final'] = scr['screen_pass'].astype(int) * scr['score_fund']
    scr = QuantileSorter('score_final', 5).assign_quantiles(scr)
    scr_test = scr[scr['date'] >= pd.Timestamp(test_start)].copy()
    scr_returns = q_builder.build(scr_test)
    scr_ls = q_builder.long_short(scr_returns, 5, 1).rename('screening_ls')

    pred_features = [col for col in expected_return_features if col in df.columns]
    pred_model = expected_return_model_class(
        feature_cols=pred_features,
        target_col=FORWARD_RETURN_COL,
        train_end=train_end,
    )
    pred_model.fit(df)
    pred_df = pred_model.predict_panel(df)
    pred_df = QuantileSorter('pred_ret_1m_fwd', 5).assign_quantiles(pred_df)
    pred_test = pred_df[pred_df['date'] >= pd.Timestamp(test_start)].copy()
    pred_returns = q_builder.build(pred_test)
    pred_ls = q_builder.long_short(pred_returns, 5, 1).rename('prediction_ls')

    metrics = pd.concat([
        PerformanceMetrics(mp_ls).summary().assign(strategy='mispricing_ls'),
        PerformanceMetrics(scr_ls).summary().assign(strategy='screening_ls'),
        PerformanceMetrics(pred_ls).summary().assign(strategy='prediction_ls'),
    ], ignore_index=True)

    return {
        'fair_value_model': fv_model,
        'fair_value_panel': fv_df,
        'mispricing_panel': ens,
        'mispricing_quintile_returns': q_returns,
        'mispricing_ls': mp_ls,
        'screening_panel': scr,
        'screening_quintile_returns': scr_returns,
        'screening_ls': scr_ls,
        'prediction_model': pred_model,
        'prediction_panel': pred_df,
        'prediction_quintile_returns': pred_returns,
        'prediction_ls': pred_ls,
        'metrics': metrics,
    }

# Esempio:
# results = run_full_experiment(df, fair_value_model_class=PeerImpliedGBRT)
# results['metrics']

### Interpretazione cauta delle metriche out-of-sample

Alcuni alpha annualizzati e Sharpe del laboratorio sono stimati su circa 40-60 osservazioni mensili out-of-sample: numeri come un alpha annualizzato di pochi punti percentuali, positivo o negativo, possono non essere statisticamente distinguibili da zero. In universi piccoli e concentrati, `R2_OS`, Sharpe e drawdown sono molto sensibili a pochi outlier, reverse split e cambi di regime. Per questo la lettura corretta non è “un numero vince”, ma stabilità su sottoperiodi, bootstrap, information coefficient e coerenza economica del ranking. L'ML più pesante resta limitato dalla dimensione della cross-section: l'estensione naturale è importare CSV regolamentari completi, armonizzare peer europei e aggiungere fattori AQR/Fama-French o un modello di regime settoriale più robusto.
